# Notebook 10 — เตรียมข้อมูล v2: ซ่อมวันที่ด้วยหลักฐาน, panel เป้าหมาย, feature ที่รู้ก่อน issue และตรวจ leakage

**เป้าหมาย:** สร้างชุดข้อมูลที่ตรวจสอบย้อนกลับได้สำหรับการทดลองพยากรณ์ v2 (notebook 11) ที่จะทดสอบ feature เทศกาล/วันหยุด, อากาศ, ฤดูกาล, ช่วงเวลา (6 ชม.) และจราจร — **notebook นี้ไม่ fit โมเดล และไม่อ้างว่าโมเดลใดดีกว่า baseline**

**Data flow:** v1 (อ่านอย่างเดียว: `data/raw/accidents` + `data/interim/row_audit.csv`, `cleaned_events.csv`, `quarantine.csv`, `possible_undercount_flags.csv`, `reconciliation.csv` + `data/processed`) และผล notebook 09 (อ่านอย่างเดียว: `reports/v2/sources/*` + `data/external_v2/derived/*` + XLSX คู่ข้อมูลที่ลงทะเบียน) → A ตรวจ hash ทุก input (ล้มทันทีถ้าถูกแก้) → B ประกอบ lineage รายแถว + ซ่อมวันที่กำกวม → C สร้างธง unknown label ใหม่ → D panel 77 จังหวัด × 1826 วัน และ × 4 บล็อก 6 ชม. → E feature as-of → F matrix ความพร้อม + negative controls → G สถิติเชิงพรรณนา + PNG → H รายงานไทย + manifest

**การเลือก schema:** keys `province_key` (ชื่อจังหวัด v1 77 จังหวัด, join กับจุด 09 ผ่าน alias ที่ตรวจแล้วแบบตรงตัวเท่านั้น), `target_date` (วันท้องถิ่น Asia/Bangkok), `time_block` (00-06/06-12/12-18/18-24); target `y_recorded_count`; mask `training_eligible`. ไฟล์ panel กับ feature แยกกันแต่เรียงแถวตรงกันทุกแถว

**ข้อจำกัดสำคัญ:** จำนวนเป็นบันทึก MOT ที่ถูกเลือก (ไม่ใช่อุบัติเหตุทั้งหมด); การซ่อมวันที่อิงคู่ข้อมูลของผู้เผยแพร่ (ไม่ใช่ ground truth อิสระ); อากาศ/จราจร/ปฏิทินมีข้อสมมติเรื่องเวลาเผยแพร่; ไม่มีเครือข่ายใน notebook นี้; production_ready = false, operational = BLOCKED

รันได้ทั้งแบบ VS Code Run All (kernel ใหม่, ค่า default ของ root) และผ่าน `backend/scripts/run_v2_notebooks.py --stage prepare` (nbclient ด้วย `.venv` เดียวกัน)

In [1]:
# FUNCTIONS (notebook 10, part 1: I/O + dependency hashes, v1 date/time parsing, independent publisher-counterpart context, row repair, unknown-label flags) - self-contained; tests execute both FUNCTIONS cells in order (no network anywhere in notebook 10)
import datetime as dt, gzip, hashlib, json, math, os, re
from pathlib import Path
import numpy as np, pandas as pd

EXCEL_ORIGIN = dt.date(1899, 12, 30)
DISPOSITIONS = ('retained', 'collapsed', 'quarantined', 'diagnostic')
OWN_REASON_PREFIXES = ('date_', 'time_', 'coord_', 'province_missing', 'no_acc_code')
REPAIRED = 'repaired_publisher_counterpart'


class DependencyError(RuntimeError):
    """An input is missing or differs from the hash its producer recorded (tampering, a rerun or a stale copy): stop, never guess."""


# ---------- generic I/O ----------
def utc_now():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def json_safe(o):
    """Strict-JSON friendly copy: NaN/inf/NaT/NA -> None, numpy scalars -> Python, dates -> ISO strings."""
    if o is None or o is pd.NaT or o is pd.NA:
        return None
    if isinstance(o, dict):
        return {str(k): json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [json_safe(v) for v in o]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
        return o.isoformat()
    if isinstance(o, Path):
        return str(o)
    return o


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(json_safe(obj), indent=2, ensure_ascii=False, allow_nan=False) + '\n', 'utf-8')
    os.replace(tmp, path)


def write_csv(path, df):
    """Plain CSV, LF line endings; any infinite numeric value is refused (strict, finite aggregates only)."""
    path = Path(path)
    num = df.select_dtypes(include=[np.number])
    if len(num.columns) and np.isinf(num.to_numpy(dtype=float)).any():
        raise ValueError(f'{path.name}: infinite values')
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, lineterminator='\n')
    return path


def write_csv_gz(path, df):
    """Gzip CSV with a fixed header (mtime 0, no stored name): identical tables give identical bytes on every rerun."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'wb') as fh, gzip.GzipFile(filename='', mode='wb', fileobj=fh, mtime=0) as gz:
        gz.write(df.to_csv(index=False, lineterminator='\n').encode('utf-8'))
    os.replace(tmp, path)
    return path


def read_str_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def verify_file(path, expected_sha256, label):
    """sha256 of an input must equal the hash recorded by its producer; raises DependencyError otherwise."""
    p = Path(path)
    if not p.is_file():
        raise DependencyError(f'{label}: missing input {p}')
    got = sha256_file(p)
    if not expected_sha256 or got != expected_sha256:
        raise DependencyError(f'{label}: sha256 {got[:12]} != recorded {str(expected_sha256)[:12]} ({p}); input changed since its producer ran')
    return got


def functions_cells_source(nb_path):
    """Concatenated source of every '# FUNCTIONS' code cell in notebook order (hashed into manifests)."""
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    srcs = [''.join(c['source']) if isinstance(c['source'], list) else c['source'] for c in nb['cells'] if c['cell_type'] == 'code']
    hits = [s for s in srcs if s.lstrip().startswith('# FUNCTIONS')]
    if not hits:
        raise ValueError(f'{nb_path}: no FUNCTIONS cell')
    return '\n\n'.join(hits)


def code_cells_sha256(nb_path):
    """sha256 of every code cell source (cell index -> hash) so a reviewer can see exactly which cell changed."""
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    out = {}
    for i, c in enumerate(nb['cells']):
        if c['cell_type'] == 'code':
            s = ''.join(c['source']) if isinstance(c['source'], list) else c['source']
            out[f"{i}:{c.get('id', '')}"] = sha256_bytes(s.encode('utf-8'))
    return out


# ---------- v1 parsing (same rules as notebook 03; repeated here so notebook 10 is self-contained) ----------
def norm_text(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return ''
    return re.sub(r'\s+', ' ', str(v)).strip()


def norm_id(v):
    """Exact-ID normalisation only: whitespace collapse, integral floats ('123.0', 123.0) -> '123'. No fuzzy matching."""
    if v is None or isinstance(v, bool):
        return '' if v is None else str(v)
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        return '' if not math.isfinite(float(v)) else (str(int(v)) if float(v).is_integer() else repr(float(v)))
    s = norm_text(v)
    m = re.fullmatch(r'(\d+)\.0+', s)
    return m.group(1) if m else s


def parse_date_v1(raw, expected_year):
    """(date or None, rule, status, candidate ISO tuple) exactly as v1: BE years -543, both slash readings kept, never chooses."""
    s = norm_text(raw)
    if not s:
        return None, 'none', 'missing', ()
    if re.fullmatch(r'\d+(\.0+)?', s):
        n = int(float(s))
        if not 30000 <= n <= 60000:
            return None, 'excel_serial', 'invalid', ()
        d = EXCEL_ORIGIN + dt.timedelta(days=n)
        return (d if d.year == expected_year else None, 'excel_serial', 'ok' if d.year == expected_year else 'year_mismatch', (d.isoformat(),))
    m = re.fullmatch(r'(\d{4})-(\d{1,2})-(\d{1,2})', s)
    if m:
        try:
            d = dt.date(*map(int, m.groups()))
        except ValueError:
            return None, 'iso', 'invalid', ()
        return (d if d.year == expected_year else None, 'iso', 'ok' if d.year == expected_year else 'year_mismatch', (d.isoformat(),))
    m = re.fullmatch(r'(\d{1,2})/(\d{1,2})/(\d{4})', s)
    if m:
        a, b, y = map(int, m.groups())
        if y > 2400:
            y -= 543  # Buddhist-era year
        cands = []
        for dd, mm in ((a, b), (b, a)):
            try:
                d = dt.date(y, mm, dd)
            except ValueError:
                continue
            if d not in cands:
                cands.append(d)
        iso = tuple(sorted(c.isoformat() for c in cands))
        if not cands:
            return None, 'slash', 'invalid', ()
        if len(cands) > 1:
            return None, 'slash_dmy_vs_mdy', 'ambiguous', iso
        d = cands[0]
        return (d if d.year == expected_year else None, 'slash_equal_parts' if a == b else 'slash_single_valid_order',
                'ok' if d.year == expected_year else 'year_mismatch', iso)
    return None, 'unrecognized', 'invalid', ()


def parse_time_v1(raw):
    """(seconds since local midnight or None, status). 'H:MM[:SS]' with h < 24 (24:00 is invalid) or an Excel day fraction < 1."""
    s = norm_text(raw)
    if not s:
        return None, 'missing'
    m = re.fullmatch(r'(\d{1,2}):(\d{2})(?::(\d{2}))?', s)
    if m:
        h, mi, se = int(m.group(1)), int(m.group(2)), int(m.group(3) or 0)
        return (h * 3600 + mi * 60 + se, 'ok') if h < 24 and mi < 60 and se < 60 else (None, 'invalid')
    try:
        x = float(s)
    except ValueError:
        return None, 'invalid'
    if not math.isfinite(x) or not 0 <= x < 1:
        return None, 'invalid'
    sec = int(round(x * 86400))
    return (None, 'invalid') if sec >= 86400 else (sec, 'ok')


# ---------- independent publisher-counterpart context (province + HH:MM) ----------
def province_key(v, aliases):
    """v1 panel join key: whitespace-normalized text through the EXACT audited alias map (no fuzzy/prefix rules); '' when blank."""
    t = norm_text(v)
    return aliases.get(t, t)


def time_minute(v):
    """Minute of the local day from CSV text, a typed time/datetime or an Excel day fraction; None when unparseable or >= 24:00."""
    if v is None or isinstance(v, bool):
        return None
    if isinstance(v, dt.datetime):
        v = v.time()
    if isinstance(v, dt.time):
        return v.hour * 60 + v.minute
    if isinstance(v, (int, float, np.integer, np.floating)):
        x = float(v)
        if not math.isfinite(x) or not 0 <= x < 1:
            return None
        sec = int(round(x * 86400))
        return None if sec >= 86400 else sec // 60
    sec, _ = parse_time_v1(v)
    return None if sec is None else sec // 60


def read_xlsx_context(path, id_header, date_header, fields, max_header_scan=30):
    """(DataFrame acc_code + RAW context cells, meta) from the first sheet whose header row holds both id_header and date_header.
    Headers are compared with all whitespace removed (as notebook 09). A missing context header gives None cells (unavailable)."""
    from openpyxl import load_workbook
    wb = load_workbook(path, read_only=True, data_only=True)
    try:
        for ws in wb.worksheets:
            it = ws.iter_rows(values_only=True)
            for i, row in enumerate(it):
                if i >= max_header_scan:
                    break
                names = [re.sub(r'\s+', '', str(c)) if c is not None else '' for c in (row or ())]
                if id_header in names and date_header in names:
                    ci = names.index(id_header)
                    cols = {f['field']: names.index(f['header']) for f in fields if f['header'] in names}
                    recs = []
                    for r in it:
                        if r is None or all(c is None or (isinstance(c, str) and not c.strip()) for c in r):
                            continue
                        rec = {'acc_code': norm_id(r[ci] if ci < len(r) else None)}
                        for f in fields:
                            j = cols.get(f['field'])
                            rec[f['field']] = None if j is None or j >= len(r) else r[j]
                        recs.append(rec)
                    return pd.DataFrame(recs, columns=['acc_code'] + [f['field'] for f in fields]), {
                        'sheet': ws.title, 'header_row': i + 1, 'rows': len(recs), 'missing_headers': [f['header'] for f in fields if f['field'] not in cols]}
        raise ValueError(f'{path}: no header row with {id_header!r} and {date_header!r}')
    finally:
        wb.close()


CTX10_STATUSES = ('agree', 'conflict_province', 'conflict_time', 'unavailable_xlsx_id', 'unavailable_province', 'unavailable_time',
                  'unavailable_xlsx_file')


def independent_context(rows, xctx, aliases):
    """Per CSV row (columns acc_code, province, time_raw): 'agree' only when EVERY XLSX row of the exact ID has a non-blank province key
    equal to the CSV province key (audited aliases applied on both sides) AND a parseable HH:MM equal to the CSV HH:MM. Any difference ->
    conflict_*; anything missing on either side -> unavailable_* (never silently treated as verified). xctx None -> unavailable_xlsx_file."""
    if xctx is None:
        return pd.Series(['unavailable_xlsx_file'] * len(rows), index=rows.index, dtype=object)
    g = {}
    for a, p, t in zip(xctx['acc_code'].map(norm_id), xctx['province'], xctx['time']):
        if a == '':
            continue
        s = g.setdefault(a, (set(), set()))
        s[0].add(province_key(p, aliases))
        tm = time_minute(t)
        s[1].add(-1 if tm is None else tm)
    out = []
    for a, p, t in zip(rows['acc_code'].map(norm_id), rows['province'], rows['time_raw']):
        if a not in g:
            out.append('unavailable_xlsx_id')
            continue
        pks, tms = g[a]
        ck, cm = province_key(p, aliases), time_minute(t)
        prov_ok = None if (ck == '' or '' in pks) else (pks == {ck})
        time_ok = None if (cm is None or -1 in tms) else (tms == {cm})
        if prov_ok is False:
            out.append('conflict_province')
        elif time_ok is False:
            out.append('conflict_time')
        elif prov_ok is None:
            out.append('unavailable_province')
        elif time_ok is None:
            out.append('unavailable_time')
        else:
            out.append('agree')
    return pd.Series(out, index=rows.index, dtype=object)


# ---------- row repair and v2 dispositions ----------
def own_reasons(reasons):
    """Row-level invalidity tokens of a v1 reasons string (group-derived reasons such as id_group_member_invalid / id_conflict_* dropped)."""
    return [t for t in str(reasons).split(';') if t and t.startswith(OWN_REASON_PREFIXES)]


def date_decisions(rows, xlsx_years_ce, require_ctx=True, accept_partial=False):
    """rows: v1 row_audit + source_year_ce, ev09_status, ev09_resolved_date, ctx10_status. Returns (date_decision, repaired_date).
    Only a v1 'ambiguous' row may be repaired: 09 resolved it, the date is one of its own candidates and in its file year, and the independent
    context agrees (partial context accepted only when explicitly configured). v1 unambiguous dates are never changed."""
    dec, rep = [], []
    for disp, st, cands, y, s09, d09, c10 in zip(rows['disposition'], rows['date_status'], rows['date_candidates'], rows['source_year_ce'],
                                                rows['ev09_status'], rows['ev09_resolved_date'], rows['ctx10_status']):
        d = None
        if disp == 'diagnostic':
            k = 'diagnostic_not_label'
        elif st != 'ambiguous':
            k = 'v1_date_kept' if st == 'ok' else f'v1_date_{st}'
        elif int(y) not in xlsx_years_ce:
            k = 'ambiguous_no_xlsx_year'
        elif not s09:
            k = 'ambiguous_no_09_evidence_row'
        elif s09 != 'resolved':
            k = f'ambiguous_09_{s09}'
        elif d09 not in str(cands).split(';'):
            k = 'ambiguous_09_date_not_a_candidate'
        elif int(str(d09)[:4]) != int(y):
            k = 'ambiguous_09_date_out_of_year'
        elif require_ctx and c10 != 'agree' and not (accept_partial and c10 in ('unavailable_province', 'unavailable_time')):
            k = f'ambiguous_ctx10_{c10}'
        else:
            k, d = REPAIRED, d09
        dec.append(k)
        rep.append(d)
    return pd.Series(dec, index=rows.index, dtype=object), pd.Series(rep, index=rows.index, dtype=object)


def redecide_groups(rows, signatures):
    """v2 disposition per row. rows: source_year, source_row (int), acc_code, disposition, reasons, kept_source_row, date_decision.
    signatures: {(source_year, source_row): (sig_all, sig_time_location)} for members of repaired groups (v1 compare fields with the repaired date).
    Groups (source_year, acc_code) WITHOUT a repaired member keep v1 disposition/reasons/kept_source_row exactly (id_conflict_* stay; no groupby-first).
    Groups WITH a repaired member are re-decided with the v1 rule: any remaining own reason -> whole group quarantined (members without own reasons
    get id_group_member_invalid); else size 1 -> retained; identical signatures -> lowest source_row retained, others collapsed; differing
    signatures -> quarantined id_conflict_time_location (event key differs) or id_conflict_other_fields."""
    out = rows[['disposition', 'reasons', 'kept_source_row']].copy()
    out.columns = ['v2_disposition', 'v2_reasons', 'v2_kept_source_row']
    out['v2_redecided'] = False
    has_id = rows['acc_code'].astype(str).ne('') & rows['disposition'].ne('diagnostic')
    rep_groups = set(zip(rows.loc[has_id & rows['date_decision'].eq(REPAIRED), 'source_year'],
                         rows.loc[has_id & rows['date_decision'].eq(REPAIRED), 'acc_code']))
    if not rep_groups:
        return out
    sub = rows[has_id & pd.Series(list(zip(rows['source_year'], rows['acc_code'])), index=rows.index).isin(rep_groups)]
    for (y, a), g in sub.groupby(['source_year', 'acc_code'], sort=True):
        own = {i: [t for t in own_reasons(r) if not (t == 'date_ambiguous' and dd == REPAIRED)]
               for i, r, dd in zip(g.index, g['reasons'], g['date_decision'])}
        out.loc[g.index, 'v2_redecided'] = True
        out.loc[g.index, 'v2_kept_source_row'] = ''
        if any(own.values()):
            for i in g.index:
                out.at[i, 'v2_disposition'] = 'quarantined'
                out.at[i, 'v2_reasons'] = ';'.join(own[i]) if own[i] else 'id_group_member_invalid'
            continue
        if len(g) == 1:
            out.loc[g.index, ['v2_disposition', 'v2_reasons']] = ['retained', '']
            continue
        sigs = [signatures.get((y, int(r))) for r in g['source_row']]
        if any(s is None for s in sigs):
            raise KeyError(f'missing raw signature for repaired group {y}:{a}')
        if len({s[0] for s in sigs}) == 1:
            first = int(g['source_row'].astype(int).min())
            for i, r in zip(g.index, g['source_row'].astype(int)):
                out.at[i, 'v2_disposition'] = 'retained' if r == first else 'collapsed'
                out.at[i, 'v2_reasons'] = ''
                out.at[i, 'v2_kept_source_row'] = str(first)
        else:
            reason = 'id_conflict_time_location' if len({s[1] for s in sigs}) > 1 else 'id_conflict_other_fields'
            out.loc[g.index, ['v2_disposition', 'v2_reasons']] = ['quarantined', reason]
    return out


def v1_signatures(raw, aliases, source_rows, event_dates, bbox):
    """Signature over the v1 compare fields (every schema alias except date_raw/time_raw/latitude/longitude, plus event_date, time_sec, lat, lon)
    for the given 1-based source rows of ONE raw CSV. event_dates: {source_row: ISO date used for the event key}. Returns {source_row: (all, tl)}."""
    out = {}
    hdr = {canon: next((n for n in names if n in raw.columns), None) for canon, names in aliases.items()}
    compare = [k for k in aliases if k not in ('date_raw', 'time_raw', 'latitude', 'longitude')]
    for sr in source_rows:
        r = raw.iloc[int(sr) - 2]
        vals = [norm_text(r[hdr[k]]) if hdr[k] else '' for k in compare]
        sec, _ = parse_time_v1(r[hdr['time_raw']]) if hdr['time_raw'] else (None, 'missing')
        la = pd.to_numeric(norm_text(r[hdr['latitude']]), errors='coerce') if hdr['latitude'] else np.nan
        lo = pd.to_numeric(norm_text(r[hdr['longitude']]), errors='coerce') if hdr['longitude'] else np.nan
        la, lo = (float(la) if np.isfinite(la) else np.nan), (float(lo) if np.isfinite(lo) else np.nan)
        tl = '\x1f'.join(map(str, (event_dates[int(sr)], sec, la, lo)))
        out[int(sr)] = ('\x1f'.join(vals) + '\x1f' + tl, tl)
    return out


# ---------- unknown-label flags ----------
def unknown_flags(x, provinces, panel_dates, year_days=None, policy='v1_count_only', all_token='*ALL*'):
    """x: rows needing flags with columns province_key, flag_dates (';' ISO), reasons, source_year_ce, flag_kind.
    Each row flags every one of its candidate dates inside the panel for its province; a blank or non-panel province flags ALL panel
    provinces. Out-of-panel dates are counted, never flagged. Rows without any candidate date: policy v1_count_only -> counted per
    province-year (unplaceable table); flag_province_year -> every day of that year for the province (all provinces when unknown).
    Returns (flags[province_key,date,n_flagged_rows,flag_reasons], unplaceable[province_key,year_ce,n_rows], stats)."""
    provinces = list(provinces)
    pset, dset = set(provinces), set(panel_dates)
    recs, unpl = [], []
    st = {'rows_in': len(x), 'rows_all_provinces': 0, 'dates_outside_panel': 0, 'rows_without_candidate_date': 0}
    for pk, ds, rs, y, kind in zip(x['province_key'], x['flag_dates'], x['reasons'], x['source_year_ce'], x['flag_kind']):
        targets = [pk] if pk in pset else provinces
        st['rows_all_provinces'] += pk not in pset
        tag = ';'.join(t for t in [kind] + str(rs).split(';') if t)
        dates = [d for d in str(ds).split(';') if d]
        if not dates:
            st['rows_without_candidate_date'] += 1
            unpl.append((pk if pk in pset else all_token, int(y)))
            if policy == 'flag_province_year':
                dates = [d for d in (year_days or {}).get(int(y), [])]
            elif policy != 'v1_count_only':
                raise ValueError(f'unknown no_candidate_date_policy {policy!r}')
        for d in dates:
            if d not in dset:
                st['dates_outside_panel'] += 1
                continue
            recs.extend((p, d, tag) for p in targets)
    cols = ['province_key', 'date', 'n_flagged_rows', 'flag_reasons']
    if recs:
        f = pd.DataFrame(recs, columns=['province_key', 'date', 'tag'])
        flags = f.groupby(['province_key', 'date'], sort=True)['tag'].agg(
            n_flagged_rows='size', flag_reasons=lambda r: ';'.join(sorted({t for v in r for t in v.split(';') if t}))).reset_index()[cols]
    else:
        flags = pd.DataFrame(columns=cols)
    up = (pd.DataFrame(unpl, columns=['province_key', 'year_ce']).value_counts().rename('n_rows').reset_index()
          .sort_values(['year_ce', 'province_key']).reset_index(drop=True)) if unpl else pd.DataFrame(columns=['province_key', 'year_ce', 'n_rows'])
    st['flagged_province_days'] = len(flags)
    return flags, up, st


def expand_v1_flags(v1flags, provinces, all_token='*ALL*'):
    """v1 possible_undercount_flags with '*ALL*' expanded to every panel province -> set of (province, date)."""
    out = set()
    for p, d in zip(v1flags['province'], v1flags['date']):
        out.update(((q, d) for q in provinces) if p == all_token else [(p, d)])
    return out


In [2]:
# FUNCTIONS (notebook 10, part 2: time blocks + panels, festival/bank-calendar/season/weather/traffic as-of features, allowlist guard, leakage negative controls, descriptive helpers)
def offset_str(utc_offset_seconds):
    s = int(utc_offset_seconds)
    sign = '+' if s >= 0 else '-'
    return f'{sign}{abs(s) // 3600:02d}:{abs(s) % 3600 // 60:02d}'


def fmt_local(ts, utc_offset_seconds):
    """Naive local timestamps -> ISO strings with the explicit local offset (e.g. 2021-01-01T00:00:00+07:00)."""
    return pd.DatetimeIndex(ts).strftime('%Y-%m-%dT%H:%M:%S') + offset_str(utc_offset_seconds)


def time_block_index(time_sec, block_hours=6):
    """Block of a valid local time of day: [b*h, (b+1)*h) hours; 06:00:00 belongs to 06-12. Invalid (None, <0, >=24 h) raises."""
    if time_sec is None or (isinstance(time_sec, float) and math.isnan(time_sec)):
        raise ValueError('event time unknown: such rows are not retained')
    s = int(time_sec)
    if not 0 <= s < 86400:
        raise ValueError(f'time_sec {s} outside one local day')
    return s // (block_hours * 3600)


def block_label(b, block_hours=6):
    return f'{b * block_hours:02d}-{(b + 1) * block_hours:02d}'


def split_of(dates, splits):
    """Split name per date from inclusive [start, end] ranges; a date outside every split raises (no silent labels)."""
    d = pd.DatetimeIndex(dates)
    out = pd.Series([None] * len(d), dtype=object)
    for s in splits:
        m = (d >= pd.Timestamp(s['start'])) & (d <= pd.Timestamp(s['end']))
        if (out[m].notna()).any():
            raise ValueError(f"split {s['name']} overlaps another split")
        out[m] = s['name']
    if out.isna().any():
        raise ValueError('dates outside every split')
    return out.to_numpy()


def build_panels(events, provinces, dates, flags, unplaceable, splits, issue_hour, days_before, utc_offset, block_hours=6):
    """events: province_key, event_date (ISO), time_sec. Returns (daily, block) panels over provinces x dates (x blocks).
    y_recorded_count = selected retained records in the cell (0 in a known cell is 'no selected record', not a proven zero).
    Daily flags are broadcast to every block; training_eligible = not possibly_undercounted. issue = target day - days_before at
    issue_hour local, identical for all blocks of a day; target_start/end are explicit local [start, end)."""
    nb = 24 // block_hours
    pidx = pd.Index(list(provinces), name='province_key')
    didx = pd.DatetimeIndex(pd.to_datetime(list(dates)), name='target_date')
    ev = events.assign(_d=pd.to_datetime(events['event_date']), _b=[time_block_index(t, block_hours) for t in events['time_sec']])
    if not ev['province_key'].isin(pidx).all() or not ev['_d'].isin(didx).all():
        raise ValueError('events outside the panel keys must be handled (flagged) before building panels')
    daily = pd.MultiIndex.from_product([pidx, didx]).to_frame(index=False)
    cnt = ev.groupby(['province_key', '_d']).size().rename('y_recorded_count')
    daily = daily.join(cnt, on=['province_key', 'target_date']).fillna({'y_recorded_count': 0})
    daily['y_recorded_count'] = daily['y_recorded_count'].astype(int)
    f = flags.assign(target_date=pd.to_datetime(flags['date'])).set_index(['province_key', 'target_date'])[['n_flagged_rows', 'flag_reasons']]
    daily = daily.join(f, on=['province_key', 'target_date'])
    daily['n_flagged_rows'] = daily['n_flagged_rows'].fillna(0).astype(int)
    daily['flag_reasons'] = daily['flag_reasons'].fillna('')
    daily['possibly_undercounted'] = daily['n_flagged_rows'] > 0
    daily['training_eligible'] = ~daily['possibly_undercounted']
    up = unplaceable.assign(year_ce=unplaceable['year_ce'].astype(int)) if len(unplaceable) else unplaceable
    yr = daily['target_date'].dt.year
    upn = {(p, int(y)): int(n) for p, y, n in zip(up['province_key'], up['year_ce'], up['n_rows'])} if len(up) else {}
    daily['unplaceable_rows_province_year'] = [upn.get((p, y), 0) + upn.get(('*ALL*', y), 0) for p, y in zip(daily['province_key'], yr)]
    issue = daily['target_date'] - pd.Timedelta(days=days_before) + pd.Timedelta(hours=issue_hour)
    daily['issue_time_local'] = fmt_local(issue, utc_offset)
    daily['issue_time_utc'] = (issue - pd.Timedelta(seconds=utc_offset)).dt.strftime('%Y-%m-%dT%H:%M:%S') + '+00:00'
    daily['target_start_local'] = fmt_local(daily['target_date'], utc_offset)
    daily['target_end_local'] = fmt_local(daily['target_date'] + pd.Timedelta(days=1), utc_offset)
    daily['split'] = split_of(daily['target_date'], splits)
    blk = daily.loc[daily.index.repeat(nb)].reset_index(drop=True)
    blk.insert(2, 'block_index', np.tile(np.arange(nb), len(daily)))
    blk.insert(3, 'time_block', [block_label(b, block_hours) for b in blk['block_index']])
    bc = ev.groupby(['province_key', '_d', '_b']).size().rename('_n')
    blk = blk.join(bc, on=['province_key', 'target_date', 'block_index'])
    blk['y_recorded_count'] = blk['_n'].fillna(0).astype(int)
    blk = blk.drop(columns=['_n'])
    st = blk['target_date'] + pd.to_timedelta(blk['block_index'] * block_hours, unit='h')
    blk['target_start_local'] = fmt_local(st, utc_offset)
    blk['target_end_local'] = fmt_local(st + pd.Timedelta(hours=block_hours), utc_offset)
    for df in (daily, blk):
        df['target_date'] = df['target_date'].dt.strftime('%Y-%m-%d')
    return daily, blk


def reconcile_panels(daily, blk, n_events):
    """Hard checks: non-negative integer counts, daily == sum of its blocks, total == placed events, flags broadcast, eligibility = not flagged."""
    s = blk.groupby(['province_key', 'target_date'], sort=False)['y_recorded_count'].sum()
    d = daily.set_index(['province_key', 'target_date'])['y_recorded_count']
    checks = {
        'counts_nonnegative': bool((daily['y_recorded_count'] >= 0).all() and (blk['y_recorded_count'] >= 0).all()),
        'daily_equals_sum_of_blocks': bool(s.reindex(d.index).eq(d).all()),
        'daily_total_equals_events': int(daily['y_recorded_count'].sum()) == int(n_events),
        'block_total_equals_events': int(blk['y_recorded_count'].sum()) == int(n_events),
        'blocks_per_day': bool(blk.groupby(['province_key', 'target_date']).size().eq(24 // 6 if len(blk) % 4 == 0 else -1).all()),
        'flags_broadcast_to_blocks': bool(blk.groupby(['province_key', 'target_date'])['possibly_undercounted'].nunique().eq(1).all()),
        'eligible_is_not_flagged': bool((daily['training_eligible'] == ~daily['possibly_undercounted']).all()),
    }
    bad = [k for k, v in checks.items() if not v]
    if bad:
        raise AssertionError(f'panel reconciliation failed: {bad}')
    return checks


# ---------- deterministic date features ----------
def core_date_features(dates):
    d = pd.DatetimeIndex(pd.to_datetime(list(dates)))
    diy = np.where(d.is_leap_year, 366, 365)
    ang = 2 * np.pi * (d.dayofyear - 1) / diy
    return pd.DataFrame({'target_date': d.strftime('%Y-%m-%d'), 'day_of_week': d.dayofweek, 'month': d.month, 'day_of_month': d.day,
                         'is_weekend': (d.dayofweek >= 5).astype(int), 'doy_sin': np.sin(ang), 'doy_cos': np.cos(ang)})


def festival_runs(month_days, years):
    """Contiguous occurrences (first, last) of a fixed-date festival over `years`, joined across Dec 31 -> Jan 1."""
    days = sorted({dt.date(y, m, d) for y in years for m, d in month_days})
    runs = []
    for d in days:
        if runs and (d - runs[-1][1]).days == 1:
            runs[-1][1] = d
        else:
            runs.append([d, d])
    return [tuple(r) for r in runs]


def signed_distance(d, runs):
    """target - nearest festival day: 0 inside, negative before the first day, positive after the last day; ties prefer 'before'."""
    best = None
    for a, b in runs:
        v = 0 if a <= d <= b else ((d - a).days if d < a else (d - b).days)
        if best is None or abs(v) < abs(best) or (abs(v) == abs(best) and v < best):
            best = v
    return best


def festival_features(dates, festivals, clip, windows):
    d = [dt.date.fromisoformat(str(x)[:10]) for x in dates]
    years = range(min(x.year for x in d) - 1, max(x.year for x in d) + 2)
    out = pd.DataFrame({'target_date': [x.isoformat() for x in d]})
    for f in festivals:
        runs = festival_runs(f['month_days'], years)
        raw = np.array([signed_distance(x, runs) for x in d])
        out[f"fest_{f['id']}_dist"] = np.clip(raw, -clip, clip).astype(int)
        out[f"fest_{f['id']}_in"] = (raw == 0).astype(int)
        for w, (lo, hi) in windows.items():
            out[f"fest_{f['id']}_{w}"] = ((raw >= lo) & (raw <= hi)).astype(int)
    return out


def season_features(dates, months):
    m = pd.DatetimeIndex(pd.to_datetime(list(dates))).month
    allm = sorted(x for v in months.values() for x in v)
    if allm != list(range(1, 13)):
        raise ValueError('season months must partition 1..12')
    return pd.DataFrame({'target_date': pd.DatetimeIndex(pd.to_datetime(list(dates))).strftime('%Y-%m-%d'),
                         **{f'season_{k}': np.isin(m, v).astype(int) for k, v in months.items()}})


# ---------- reviewed BOT bank calendar as of issue (same replay rules as notebook 09) ----------
def available_from_local(known_from):
    """Date-only known_from is usable from the END of that local day: known_from + 1 day at 00:00 local (naive)."""
    return dt.datetime.combine(dt.date.fromisoformat(str(known_from)) + dt.timedelta(days=1), dt.time(0))


def bank_holidays_as_of(records, issue_local):
    """Applied records only, replayed in known_from order (adds before removes on the same day); absence is not an ordinary day."""
    r = records[records['applied'].astype(bool)]
    r = r[r['known_from'].map(available_from_local) <= issue_local]
    r = r.assign(_o=r['action'].map({'add': 0, 'remove': 1})).sort_values(['known_from', '_o', 'date'])
    days = set()
    for d, a in zip(r['date'], r['action']):
        if a == 'add':
            days.add(d)
        else:
            days.discard(d)
    return days


def calendar_year_coverage(records, sources, year, issue_local):
    base = [s['id'] for s in sources if s.get('role') == 'annual_base' and year in s.get('years', [])]
    ok = records[records['applied'].astype(bool) & records['source_id'].isin(base)]
    if not len(ok):
        return 'unknown_no_applied_reviewed_annual_source'
    if ok['known_from'].map(available_from_local).min() > issue_local:
        return 'unknown_annual_source_not_yet_known'
    return 'reviewed_annual_base_known_later_amendments_unknown'


def load_calendar_records(path):
    """09 calendar_records.csv -> typed frame; 'applied' is parsed from text (never bool('False'))."""
    r = read_str_csv(path)
    bad = set(r['applied']) - {'True', 'False'}
    if bad:
        raise ValueError(f'calendar applied values {bad}')
    r['applied'] = r['applied'].eq('True')
    if not r['action'].isin(['add', 'remove']).all():
        raise ValueError('calendar action outside add/remove')
    return r


def bank_calendar_features(dates, records, sources, issue_hour, days_before):
    """bank_holiday_asof (1/0, NaN when the target year's reviewed coverage is unknown at issue) and bank_calendar_known_asof (0/1)."""
    rows = []
    for x in dates:
        d = dt.date.fromisoformat(str(x)[:10])
        issue = dt.datetime.combine(d - dt.timedelta(days=days_before), dt.time(int(issue_hour)))
        cov = calendar_year_coverage(records, sources, d.year, issue)
        known = cov.startswith('reviewed')
        rows.append({'target_date': d.isoformat(), 'bank_holiday_asof': (1.0 if d.isoformat() in bank_holidays_as_of(records, issue) else 0.0) if known else np.nan,
                     'bank_calendar_known_asof': int(known), 'bank_calendar_coverage': cov})
    return pd.DataFrame(rows)


# ---------- archived weather (JMA GSM previous_day2) ----------
def weather_features(wx, code_to_key, buffer_hours, issue_hour, days_before, utc_offset, prefix):
    """09 aggregate -> (features, audit). Usable only when the variable is complete (09 *_missing False) AND both guards hold: 09 issue_bound_ok
    and the independent 10 check latest_issue_bound_utc + buffer <= (target day - days_before) at issue_hour local. A 09 'ok' that fails the
    10 recomputation raises (inconsistent guard). Unusable -> NaN (never 0, never a partial sum, never target-day observations)."""
    keys = ['province_key', 'target_date', 'time_block']
    if wx is None or not len(wx):
        return pd.DataFrame(columns=keys + [f'{prefix}rain_sum_mm_d2', f'{prefix}temp_mean_c_d2']), {'rows': 0}
    w = wx.copy()
    for c in ('rain_missing', 'temp_missing', 'issue_bound_ok'):
        w[c] = w[c].astype(str).eq('True')
    w['province_key'] = w['province_code'].astype(float).astype(int).map(code_to_key)
    if w['province_key'].isna().any():
        raise DependencyError('weather province_code without a province point')
    w['target_date'] = pd.to_datetime(w['date']).dt.strftime('%Y-%m-%d')
    issue_utc = (pd.to_datetime(w['target_date']) - pd.Timedelta(days=days_before) + pd.Timedelta(hours=issue_hour)
                 - pd.Timedelta(seconds=utc_offset)).dt.tz_localize('UTC')
    bound = pd.to_datetime(w['latest_issue_bound_utc'], utc=True, errors='coerce')
    ok10 = (bound + pd.Timedelta(hours=buffer_hours)) <= issue_utc
    ok10 = ok10.fillna(False).astype(bool)
    if (w['issue_bound_ok'] & ~ok10).any():
        raise DependencyError(f"weather: {int((w['issue_bound_ok'] & ~ok10).sum())} rows pass the 09 guard but fail the independent 10 guard")
    if w.duplicated(keys).any():
        raise DependencyError('weather: duplicate province/date/block rows')
    usable = ok10 & w['issue_bound_ok']
    rain = pd.to_numeric(w['rain_sum'], errors='coerce').where(usable & ~w['rain_missing'])
    temp = pd.to_numeric(w['temp_mean'], errors='coerce').where(usable & ~w['temp_missing'])
    if (rain < 0).any() or np.isinf(rain).any() or np.isinf(temp).any():
        raise DependencyError('weather: negative rain or infinite value')
    out = pd.DataFrame({'province_key': w['province_key'], 'target_date': w['target_date'], 'time_block': w['time_block'],
                        f'{prefix}rain_sum_mm_d2': rain, f'{prefix}temp_mean_c_d2': temp})
    lead_h = (issue_utc - bound).dt.total_seconds() / 3600
    audit = {'rows': len(w), 'rows_guard_ok': int(usable.sum()), 'rows_rain_complete_and_guard_ok': int(rain.notna().sum()),
             'rows_temp_complete_and_guard_ok': int(temp.notna().sum()), 'rows_guard_failed': int((~usable).sum()),
             'issue_minus_latest_bound_hours_min': float(lead_h[usable].min()) if usable.any() else None,
             'issue_minus_latest_bound_hours_median': float(lead_h[usable].median()) if usable.any() else None,
             'issue_minus_latest_bound_hours_max': float(lead_h[usable].max()) if usable.any() else None}
    return out, audit


# ---------- DOH AADT as of issue ----------
def traffic_vintage_by_date(meta, dates, issue_hour, days_before, utc_offset):
    """Newest PRIOR survey year (survey_year_ce < target year) among normalized resources whose proxy available_from <= issue time (UTC).
    No target-year actuals, no future or nearest backfill; NaN when nothing qualifies."""
    m = meta[meta['normalize_status'].astype(str).eq('normalized')].copy()
    m['survey_year_ce'] = m['survey_year_ce'].astype(int)
    m['avail'] = pd.to_datetime(m['available_from_proxy_utc'], utc=True, errors='coerce')
    m = m[m['avail'].notna()]
    rows = []
    for x in dates:
        d = pd.Timestamp(str(x)[:10])
        issue_utc = (d - pd.Timedelta(days=days_before) + pd.Timedelta(hours=issue_hour) - pd.Timedelta(seconds=utc_offset)).tz_localize('UTC')
        ok = m[(m['survey_year_ce'] < d.year) & (m['avail'] <= issue_utc)]
        v = ok.sort_values('survey_year_ce').iloc[-1] if len(ok) else None
        rows.append({'target_date': d.strftime('%Y-%m-%d'), 'trf_vintage_survey_year_ce': int(v['survey_year_ce']) if v is not None else np.nan,
                     'trf_vintage_available_from_utc': v['avail'].isoformat() if v is not None else None})
    return pd.DataFrame(rows)


def traffic_features(summary, vintages, provinces):
    """(province, date) features from the chosen vintage's province summary; a province absent from that vintage stays NaN."""
    s = summary.copy()
    s['survey_year_ce'] = s['survey_year_ce'].astype(int)
    s = s.rename(columns={'mean_total_aadt': 'trf_mean_aadt', 'median_total_aadt': 'trf_median_aadt',
                          'mean_heavy_share_pct': 'trf_mean_heavy_share_pct', 'n_station_rows': 'trf_n_station_rows'})
    feats = ['trf_mean_aadt', 'trf_median_aadt', 'trf_mean_heavy_share_pct', 'trf_n_station_rows']
    for c in feats:
        s[c] = pd.to_numeric(s[c], errors='coerce')
    if s.duplicated(['province_key', 'survey_year_ce']).any():
        raise DependencyError('traffic summary: duplicate province/survey year')
    grid = pd.MultiIndex.from_product([list(provinces), vintages['target_date']], names=['province_key', 'target_date']).to_frame(index=False)
    grid = grid.merge(vintages, on='target_date', how='left')
    grid = grid.merge(s[['province_key', 'survey_year_ce'] + feats], left_on=['province_key', 'trf_vintage_survey_year_ce'],
                      right_on=['province_key', 'survey_year_ce'], how='left').drop(columns=['survey_year_ce'])
    return grid


# ---------- allowlist / leakage guards ----------
def validate_feature_columns(cols, allowlist, forbidden_names, forbidden_substrings):
    """Every model-input column must be allowlisted and must not be a forbidden name or contain a forbidden substring; raises ValueError."""
    cols = list(cols)
    forb = [c for c in cols if c in set(forbidden_names) or any(s in c for s in forbidden_substrings)]
    extra = [c for c in cols if c not in set(allowlist)]
    if forb or extra:
        raise ValueError(f'feature guard: forbidden {forb}, not allowlisted {extra}')
    return True


def assert_finite_features(df, cols):
    num = df[[c for c in cols if pd.api.types.is_numeric_dtype(df[c])]]
    if len(num.columns) and np.isinf(num.to_numpy(dtype=float)).any():
        raise ValueError('infinite feature values')
    return True


def leakage_negative_controls(sources, allowlist, forbidden_names, forbidden_substrings, buffer_hours, issue_hour, days_before, utc_offset):
    """Deliberately leaky synthetic fixtures; each control must be REJECTED by the production functions above. Returns a list of results."""
    res = []
    # 1. future annual traffic: same-year survey and a prior survey revised after the issue time
    meta = pd.DataFrame({'survey_year_ce': [2023, 2024], 'normalize_status': ['normalized'] * 2,
                         'available_from_proxy_utc': ['2024-06-01T00:00:00+00:00', '2024-01-02T00:00:00+00:00']})
    v = traffic_vintage_by_date(meta, ['2024-03-01'], issue_hour, days_before, utc_offset)
    res.append({'control': 'traffic_future_or_same_year_vintage', 'rejected': bool(pd.isna(v['trf_vintage_survey_year_ce'].iloc[0]))})
    # 2. special bank holiday announced late (known_from = target day) and a later removal
    rec = pd.DataFrame({'date': ['2024-05-10', '2024-05-01', '2024-05-01'], 'action': ['add', 'add', 'remove'],
                        'known_from': ['2024-05-10', '2023-07-26', '2024-05-02'], 'applied': [True, True, True],
                        'source_id': ['late_special', 'base_2024', 'late_removal']})
    src = [{'id': 'base_2024', 'role': 'annual_base', 'years': [2024]}]
    f = bank_calendar_features(['2024-05-10', '2024-05-01'], rec, src, issue_hour, days_before)
    res.append({'control': 'bank_holiday_late_announcement', 'rejected': bool(f['bank_holiday_asof'].iloc[0] == 0.0)})
    res.append({'control': 'bank_holiday_removal_after_issue_not_applied_early', 'rejected': bool(f['bank_holiday_asof'].iloc[1] == 1.0)})
    f2 = bank_calendar_features(['2025-02-01'], rec, src, issue_hour, days_before)
    res.append({'control': 'unknown_calendar_not_ordinary_day', 'rejected': bool(pd.isna(f2['bank_holiday_asof'].iloc[0]))})
    # 3. weather whose latest hour bound + buffer is after the issue time
    issue_utc = pd.Timestamp('2024-05-09 18:00') - pd.Timedelta(seconds=utc_offset)
    wx = pd.DataFrame({'province_code': [10], 'date': ['2024-05-10'], 'time_block': ['day'], 'rain_sum': [5.0], 'temp_mean': [30.0],
                       'rain_missing': ['False'], 'temp_missing': ['False'], 'issue_bound_ok': ['False'],
                       'latest_issue_bound_utc': [str(issue_utc - pd.Timedelta(hours=buffer_hours) + pd.Timedelta(hours=1))]})
    out, _ = weather_features(wx, {10: 'X'}, buffer_hours, issue_hour, days_before, utc_offset, 'wx_')
    res.append({'control': 'weather_hour_after_issue', 'rejected': bool(out['wx_rain_sum_mm_d2'].isna().all())})
    wx2 = wx.assign(issue_bound_ok=['True'])
    try:
        weather_features(wx2, {10: 'X'}, buffer_hours, issue_hour, days_before, utc_offset, 'wx_')
        ok = False
    except DependencyError:
        ok = True
    res.append({'control': 'weather_09_guard_tampered_true', 'rejected': ok})
    # 4. outcome / label / mask / id columns offered as features
    for bad in ('accident_type', 'y_recorded_count', 'training_eligible', 'event_uid', 'target_date', 'wx_rain_missing', 'deaths'):
        try:
            validate_feature_columns(list(allowlist) + [bad], list(allowlist) + [bad], forbidden_names, forbidden_substrings)
            ok = False
        except ValueError:
            ok = True
        res.append({'control': f'allowlist_rejects_{bad}', 'rejected': ok})
    return res


# ---------- descriptive helpers (recorded selected counts; not risk, not causal effects) ----------
def dispersion(df, by):
    e = df[df['training_eligible']]
    g = e.groupby(by)['y_recorded_count']
    out = pd.DataFrame({'n_cells': df.groupby(by).size(), 'n_eligible': g.size(), 'events_eligible': g.sum(), 'mean': g.mean(),
                        'variance': g.var(ddof=1), 'zero_share': g.apply(lambda s: float((s == 0).mean())),
                        'n_nonzero': g.apply(lambda s: int((s > 0).sum())), 'max': g.max()})
    out['variance_to_mean'] = out['variance'] / out['mean'].where(out['mean'] > 0)
    return out.reset_index()


def save_png(fig, path):
    """Deterministic-ish PNG: no software/creation metadata."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=110, bbox_inches='tight', metadata={'Software': None})
    return path


In [3]:
# SETUP: roots, configs and write guards (env overrides come from backend/scripts/run_v2_notebooks.py; VS Code Run All uses the defaults)
import sys


def find_backend():
    """ROADAWARE_BACKEND_ROOT, else the first cwd ancestor that is (or contains) backend/ with config/feature_pipeline_v2.json + notebooks/."""
    env = os.environ.get('ROADAWARE_BACKEND_ROOT')
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        for cand in (d, d / 'backend'):
            if (cand / 'config' / 'feature_pipeline_v2.json').is_file() and (cand / 'notebooks').is_dir():
                return cand
    raise FileNotFoundError('backend root not found from the working directory; set ROADAWARE_BACKEND_ROOT')


def env_path(name, default):
    v = os.environ.get(name)
    return Path(v).expanduser().resolve() if v else Path(default).resolve()


def inside(p, root):
    p, root = Path(p).resolve(), Path(root).resolve()
    return p == root or root in p.parents


BACKEND = find_backend()
FCFG_PATH = BACKEND / 'config' / 'feature_pipeline_v2.json'
FCFG = json.loads(FCFG_PATH.read_text('utf-8'))
SCFG_PATH = BACKEND / FCFG['sources_config']
SCFG = json.loads(SCFG_PATH.read_text('utf-8'))
V1CFG_PATH = BACKEND / FCFG['v1_config']
V1CFG = json.loads(V1CFG_PATH.read_text('utf-8'))
NB09 = BACKEND / FCFG['notebook09']
NB10 = BACKEND / 'notebooks' / '10_v2_preparation_features_leakage.ipynb'
V1 = env_path('ROADAWARE_V1_DATA_ROOT', BACKEND)                                     # read-only
EXT = env_path('ROADAWARE_V2_EXTERNAL_ROOT', BACKEND / SCFG['default_external_root'])  # read-only for notebook 10
OUT = env_path('ROADAWARE_V2_OUTPUT_ROOT', BACKEND)
SRC = env_path('ROADAWARE_V2_SOURCES_DIR', OUT / SCFG['report_subdir'])               # read-only notebook 09 reports
DATA = env_path('ROADAWARE_V2_DATA_ROOT', OUT / FCFG['data_subdir'])                  # gitignored matrices
REP = OUT / FCFG['report_subdir']                                                     # tracked aggregate evidence
P = FCFG['panel']
ALIASES = SCFG['province_aliases']['map']
V1_PROTECTED = [V1 / 'data' / 'raw', V1 / 'data' / 'interim', V1 / 'data' / 'processed', V1 / 'data' / 'model', V1 / 'reports' / 'data',
                V1 / 'reports' / 'models']
for _name, _p in (('v2 data root', DATA), ('report dir', REP)):
    _hit = [str(v) for v in V1_PROTECTED + [EXT, SRC] if inside(_p, v) or inside(v, _p)]
    if _hit:
        raise ValueError(f'{_name} {_p} overlaps read-only inputs {_hit}; notebook 10 only writes its own roots')
if not (V1 / 'data' / 'raw' / 'accidents' / 'source_manifest.json').is_file():
    raise FileNotFoundError(f'v1 raw snapshot manifest not found under {V1}; set ROADAWARE_V1_DATA_ROOT')
if not (SRC / 'sources_manifest.json').is_file():
    raise FileNotFoundError(f'notebook 09 sources_manifest.json not found in {SRC}: run notebook 09 first (or set ROADAWARE_V2_SOURCES_DIR)')


def protected_tree_hashes(roots, base):
    return {p.relative_to(base).as_posix(): sha256_file(p) for r in roots if r.is_dir() for p in sorted(r.rglob('*')) if p.is_file()}


V1_TREE_BEFORE = protected_tree_hashes(V1_PROTECTED, V1)
DATA.mkdir(parents=True, exist_ok=True)
REP.mkdir(parents=True, exist_ok=True)
RUN = {'started_utc': utc_now(), 'sections': {}}
GAPS = []
pd.set_option('display.width', 200)
print(f'backend={BACKEND}\nv1_data_root={V1} (read-only)\nexternal_root={EXT} (read-only)\nsources_09={SRC} (read-only)\n'
      f'v2_data={DATA}\nreports={REP}\npython={sys.executable}')


backend=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend
v1_data_root=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend (read-only)
external_root=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/data/external_v2 (read-only)
sources_09=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/reports/v2/sources (read-only)
v2_data=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/data/v2
reports=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/reports/v2/preparation
python=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/.venv/bin/python


## A. ตรวจ hash ของ dependency

ตรวจว่า manifest ของ 09 เป็น schema/producer ที่คาด, config 09 และ FUNCTIONS cells ของ 09 ตรงกับที่ 09 บันทึก, ทุกไฟล์ output/derived ของ 09 ตรง hash, CSV ดิบ v1 ตรง `source_manifest.json`, ไฟล์ interim v1 ตรง `artifact_manifest.json` และ panel v1 ตรง `split_artifact_manifest.json`. ถ้าไม่ตรงจะหยุด (`DependencyError`) ไม่เดา. ไฟล์อากาศที่ 09 ยังไม่บันทึกใน manifest ถือว่า "ไม่มีแหล่ง" → feature เป็น NaN และบันทึกเป็นช่องว่าง

In [4]:
# A. Dependency hash validation (09 manifest/config/code/outputs/derived files, v1 raw + interim + processed manifests) and input loading
DEP = FCFG['dependencies']
INPUTS = []


def record_input(role, path, sha):
    INPUTS.append({'role': role, 'path': str(path), 'sha256': sha})


MAN09_PATH = SRC / 'sources_manifest.json'
MAN09 = json.loads(MAN09_PATH.read_text('utf-8'))
record_input('09_sources_manifest', MAN09_PATH, sha256_file(MAN09_PATH))
if MAN09.get('schema') != DEP['sources_manifest_schema'] or MAN09.get('producer') != DEP['sources_manifest_producer']:
    raise DependencyError(f"unexpected 09 manifest schema/producer {MAN09.get('schema')}/{MAN09.get('producer')}")
if MAN09.get('status') not in DEP['accepted_09_status']:
    raise DependencyError(f"09 status {MAN09.get('status')!r} not accepted")
record_input('09_config', SCFG_PATH, verify_file(SCFG_PATH, MAN09['config_sha256'], '09 config external_sources_v2.json'))
fn09 = sha256_bytes(functions_cells_source(NB09).encode('utf-8'))
record_input('09_functions_cells', NB09, fn09)
if DEP['require_09_functions_hash_match'] and fn09 != MAN09['functions_cells_sha256']:
    raise DependencyError('notebook 09 FUNCTIONS cells changed since the recorded 09 run; rerun/accept 09 first')
OUT09 = {Path(o['path']).name: o for o in MAN09['outputs']}
for name, o in sorted(OUT09.items()):
    record_input('09_output', SRC / name, verify_file(SRC / name, o['sha256'], f'09 output {name}'))
missing = [n for n in DEP['required_09_outputs'] if n not in OUT09]
if missing:
    raise DependencyError(f'09 manifest lacks required outputs {missing}')
DER09 = {a['path']: a for a in MAN09['cached_artifacts'] if str(a.get('resource_id', '')).startswith('derived:')}
for rel in DEP['required_09_derived'] + DEP['optional_09_derived']:
    if rel in DER09:
        record_input('09_derived', EXT / rel, verify_file(EXT / rel, DER09[rel]['sha256'], f'09 derived {rel}'))
    elif rel in DEP['required_09_derived']:
        raise DependencyError(f'09 manifest lacks required derived file {rel}')
    else:
        GAPS.append(f'09 derived {rel} not in the accepted 09 manifest: dependent features stay NaN (source unavailable)')
V1RAW_PATH = V1 / 'data' / 'raw' / 'accidents' / 'source_manifest.json'
record_input('v1_raw_manifest', V1RAW_PATH, verify_file(V1RAW_PATH, MAN09['v1_raw_manifest_sha256'], 'v1 raw source_manifest (as seen by 09)'))
V1RAW = json.loads(V1RAW_PATH.read_text('utf-8'))
for f in V1RAW['files']:
    record_input('v1_raw_csv', V1 / f['path'], verify_file(V1 / f['path'], f['sha256'], f"v1 raw {f['file']}"))
IMAN_PATH = V1 / 'data' / 'interim' / 'artifact_manifest.json'
IMAN = json.loads(IMAN_PATH.read_text('utf-8'))
if IMAN['input_source_manifest_sha256'] != sha256_file(V1RAW_PATH):
    raise DependencyError('v1 interim artifacts were not produced from this raw snapshot')
IMAN_FILES = {e['path']: e['sha256'] for e in IMAN['files']}
for rel in DEP['v1_interim_files']:
    record_input('v1_interim', V1 / rel, verify_file(V1 / rel, IMAN_FILES.get(rel), f'v1 {rel}'))
PMAN_PATH = V1 / DEP['v1_processed_manifest']
PMAN = json.loads(PMAN_PATH.read_text('utf-8'))
if {e['path']: e['sha256'] for e in PMAN['inputs']}.get('data/interim/artifact_manifest.json') != sha256_file(IMAN_PATH):
    raise DependencyError('v1 processed split manifest does not match the v1 interim manifest')
record_input('v1_panel', V1 / DEP['v1_panel'], verify_file(V1 / DEP['v1_panel'], {e['path']: e['sha256'] for e in PMAN['files']}[DEP['v1_panel']], 'v1 panel'))
for p_ in (FCFG_PATH, V1CFG_PATH):
    record_input('config', p_, sha256_file(p_))

# registered XLSX publisher counterparts (09 registry), hash-verified against the 09 date-evidence summary
DEV = json.loads((SRC / 'date_evidence_by_year.json').read_text('utf-8'))
REG09 = json.loads((EXT / 'registry.json').read_text('utf-8')) if (EXT / 'registry.json').is_file() else {'resources': {}}
XLSX = {}
for yrec in DEV['years']:
    ce, want = int(yrec['year_ce']), yrec.get('xlsx_sha256')
    if ce not in FCFG['date_repair']['xlsx_years_ce']:
        continue
    cands = [Path(e['preflight_path']) if e.get('preflight_path') and not e.get('path') else (EXT / e['path'] if e.get('path') else None)
             for e in REG09['resources'].values() if want and e.get('sha256') == want]
    path = next((c for c in cands if c is not None and c.is_file()), None)
    if path is None:
        GAPS.append(f'date repair {ce}: XLSX counterpart (sha {str(want)[:12]}) not found locally; ambiguous rows stay ambiguous (unavailable_xlsx_file)')
        continue
    record_input('09_xlsx_counterpart', path, verify_file(path, want, f'XLSX {ce}'))
    XLSX[ce] = path

# loaded inputs
AUDIT = read_str_csv(V1 / 'data' / 'interim' / 'row_audit.csv')
CLEAN = read_str_csv(V1 / 'data' / 'interim' / 'cleaned_events.csv')
QUAR = read_str_csv(V1 / 'data' / 'interim' / 'quarantine.csv')
V1FLAGS = read_str_csv(V1 / 'data' / 'interim' / 'possible_undercount_flags.csv')
V1RECON = read_str_csv(V1 / 'data' / 'interim' / 'reconciliation.csv')
V1PANEL = pd.read_csv(V1 / DEP['v1_panel'], dtype={'province': str, 'target_date': str})
EV09 = read_str_csv(EXT / 'derived' / 'date_resolution_by_id.csv')
POINTS = read_str_csv(SRC / 'province_points.csv')
CALREC = load_calendar_records(SRC / 'calendar_records.csv')
TSUM = read_str_csv(SRC / FCFG['traffic']['summary_file'])
TMETA = read_str_csv(SRC / FCFG['traffic']['metadata_file'])
WX_D = pd.read_csv(EXT / FCFG['weather']['daily_file'], dtype=str, keep_default_na=False) if FCFG['weather']['daily_file'] in DER09 else None
WX_B = pd.read_csv(EXT / FCFG['weather']['block_file'], dtype=str, keep_default_na=False) if FCFG['weather']['block_file'] in DER09 else None

pts = POINTS[POINTS['province_key'].ne('')]
PROVINCES = sorted(pts['province_key'].unique())
CODE_TO_KEY = {int(c): k for c, k in zip(pts['province_code'].astype(int), pts['province_key'])}
if len(PROVINCES) != P['expected_provinces'] or pts['province_key'].duplicated().any() or set(PROVINCES) != set(V1PANEL['province']):
    raise DependencyError(f'09 points give {len(PROVINCES)} province keys; they must equal the 77 v1 panel provinces exactly')
DATES = pd.date_range(P['start'], P['end'], freq='D').strftime('%Y-%m-%d').tolist()
assert len(DATES) == P['expected_days']
RUN['sections']['inputs'] = {'n_inputs_hashed': len(INPUTS), 'xlsx_years_available': sorted(XLSX), 'weather_daily_rows': 0 if WX_D is None else len(WX_D),
                             'weather_6h_rows': 0 if WX_B is None else len(WX_B), '09_status': MAN09['status'], '09_gaps': len(MAN09.get('gaps', []))}
write_json(REP / 'dependency_hashes.json', {'producer': FCFG['producer'], 'inputs': [dict(i, path=Path(i['path']).name) for i in INPUTS],
                                            '09_functions_cells_sha256': fn09, '09_status': MAN09['status']})
print(RUN['sections']['inputs'])


{'n_inputs_hashed': 40, 'xlsx_years_available': [2021, 2022, 2023], 'weather_daily_rows': 74460, 'weather_6h_rows': 297840, '09_status': 'completed_with_gaps', '09_gaps': 11}


## B. Lineage รายแถวและการซ่อมวันที่

เริ่มจาก `row_audit.csv` ของ v1 (ทุกแถว: source_file/source_row/event_uid) และตรวจว่าตรงกับ reconciliation/cleaned/quarantine ของ v1. แถวที่ v1 กำกวม (D/M vs M/D) ซ่อมได้เมื่อ **ทุกเงื่อนไข** ผ่าน: 09 resolved (ACC_CODE ตรงตัว, XLSX typed date เดียว, CSV raw date เดียวต่อ ID), วันที่อยู่ในชุดผู้สมัครของแถวและอยู่ในปีของไฟล์, และ notebook 10 ตรวจบริบทเองซ้ำ (จังหวัดผ่าน alias ตรงตัว + เวลา HH:MM) ว่าตรงกับ XLSX **ทุกแถวของ ID** — บริบทขาดฝั่งใดฝั่งหนึ่ง = `unavailable_*` ไม่ถือว่ายืนยัน. กลุ่ม ID ที่มีแถวซ่อมจะถูกตัดสินใหม่ด้วยกฎ v1 (ยังมีเหตุผลอื่น → กักกันทั้งกลุ่ม; หลายแถวต้องมี signature ฟิลด์ v1 ตรงกันจึง retained/collapsed ไม่เช่นนั้น id_conflict). กลุ่มอื่นคงผลของ v1 ทุกประการ (รวม id_conflict_other_fields). ปี 2026 เป็น diagnostic เสมอ. ผลรวมต่อปีต้องเท่ากับ: นำเข้า = retained + collapsed + quarantined + diagnostic

In [5]:
# B. Row lineage: reconstruct v1 decisions, independent context re-check, date repair, v2 dispositions and reconciliation
R = FCFG['date_repair']
ROWS = AUDIT.copy()
BE_TO_CE = {str(f['be_year']): int(f['ce_year']) for f in V1RAW['files']}
ROWS['source_year_ce'] = ROWS['source_year'].map(BE_TO_CE)
assert ROWS['source_year_ce'].notna().all() and ROWS['disposition'].isin(DISPOSITIONS).all()
# v1 reconciliation and lineage consistency (row_audit vs reconciliation / cleaned / quarantine)
_rc = ROWS.groupby(['source_year', 'disposition']).size().unstack(fill_value=0).reindex(columns=list(DISPOSITIONS), fill_value=0)
for r in V1RECON.itertuples():
    assert all(int(_rc.loc[r.source_year, d]) == int(getattr(r, d)) for d in DISPOSITIONS), f'v1 reconciliation differs {r.source_year}'
    assert int(r.imported_rows) == next(int(f['rows']) for f in V1RAW['files'] if str(f['be_year']) == r.source_year)
_ret = ROWS[ROWS['disposition'].eq('retained')]
assert set(_ret['event_uid']) == set(CLEAN['event_uid']) and len(_ret) == len(CLEAN)
assert set(zip(ROWS.loc[ROWS['disposition'].eq('quarantined'), 'source_year'], ROWS.loc[ROWS['disposition'].eq('quarantined'), 'source_row'])) == \
    set(zip(QUAR['source_year'], QUAR['source_row']))
# 09 per-row evidence joined on the original lineage key; acc_code and raw date must agree exactly
ev = EV09.rename(columns={'resolution_status': 'ev09_status', 'resolved_date': 'ev09_resolved_date', 'context_status': 'ev09_context_status',
                          'acc_code': 'ev09_acc_code', 'date_raw': 'ev09_date_raw'})
ev = ev[['source_year_be', 'source_row', 'ev09_acc_code', 'ev09_date_raw', 'ev09_status', 'ev09_resolved_date', 'ev09_context_status']]
if ev.duplicated(['source_year_be', 'source_row']).any():
    raise DependencyError('09 date evidence has duplicate source rows')
ROWS = ROWS.merge(ev, left_on=['source_year', 'source_row'], right_on=['source_year_be', 'source_row'], how='left', validate='one_to_one')
_j = ROWS['ev09_status'].notna()
if not ((ROWS.loc[_j, 'ev09_acc_code'].map(norm_id) == ROWS.loc[_j, 'acc_code'].map(norm_id)) & (ROWS.loc[_j, 'ev09_date_raw'] == ROWS.loc[_j, 'date_raw'])).all():
    raise DependencyError('09 date evidence lineage (acc_code/date_raw) does not match v1 row_audit')
for c in ('ev09_status', 'ev09_resolved_date', 'ev09_context_status'):
    ROWS[c] = ROWS[c].fillna('')
# independent publisher-counterpart context: only rows that could be repaired need it
ROWS['ctx10_status'] = 'not_evaluated'
CTX_META = {}
for ce in R['xlsx_years_ce']:
    m = (ROWS['source_year_ce'] == ce) & ROWS['date_status'].eq('ambiguous') & ROWS['ev09_status'].eq('resolved')
    if not m.any():
        continue
    xc = None
    if ce in XLSX:
        xc, CTX_META[ce] = read_xlsx_context(XLSX[ce], SCFG['date_evidence']['id_header'], SCFG['date_evidence']['date_header'], R['context_fields'],
                                             SCFG['date_evidence']['max_header_scan_rows'])
    ROWS.loc[m, 'ctx10_status'] = independent_context(ROWS.loc[m], xc, ALIASES)
ROWS['date_decision'], ROWS['repaired_date'] = date_decisions(ROWS, R['xlsx_years_ce'], R['require_independent_context'], R['accept_partial_context'])
ROWS['date_confidence'] = np.where(ROWS['date_decision'].eq(REPAIRED), R['confidence_label'],
                                   np.where(ROWS['date_decision'].eq('v1_date_kept'), 'v1_unambiguous_parse', 'unknown'))
# raw-row signatures for multi-row ID groups that contain a repaired row (v1 compare fields with the repaired date)
ROWS['source_row_i'] = ROWS['source_row'].astype(int)
rep = ROWS[ROWS['date_decision'].eq(REPAIRED)]
grp_size = ROWS[ROWS['acc_code'].ne('')].groupby(['source_year', 'acc_code']).size()
multi = {k for k in set(zip(rep['source_year'], rep['acc_code'])) if grp_size.get(k, 0) > 1}
SIGS = {}
for f in V1RAW['files']:
    be = str(f['be_year'])
    need = ROWS[(ROWS['source_year'] == be) & pd.Series(list(zip(ROWS['source_year'], ROWS['acc_code'])), index=ROWS.index).isin(multi)]
    if not len(need):
        continue
    raw = pd.read_csv(V1 / f['path'], dtype=str, keep_default_na=False, encoding=f['encoding'])
    assert len(raw) == int(f['rows'])
    dates = {int(sr): (d if d else parse_date_v1(dr, int(f['ce_year']))[0]) for sr, d, dr in zip(need['source_row'], need['repaired_date'], need['date_raw'])}
    dates = {k: (v.isoformat() if isinstance(v, dt.date) else v) for k, v in dates.items()}
    for sr, s in v1_signatures(raw, V1CFG['schema_aliases'], need['source_row_i'], dates, V1CFG['thailand_bbox']).items():
        SIGS[(be, sr)] = s
ROWS = ROWS.join(redecide_groups(ROWS.assign(source_row=ROWS['source_row_i']), SIGS))
# v2 event date / time / province key; v1 retained dates must be unchanged
CLEAN_D = dict(zip(CLEAN['event_uid'], CLEAN['event_date']))
ROWS['event_date_v2'] = np.where(ROWS['date_decision'].eq(REPAIRED), ROWS['repaired_date'],
                                 np.where(ROWS['date_status'].eq('ok'), ROWS['date_candidates'], ''))
ROWS.loc[ROWS['v2_disposition'].ne('retained') & ROWS['v2_disposition'].ne('collapsed') & ~ROWS['date_decision'].eq(REPAIRED) & ROWS['date_status'].ne('ok'), 'event_date_v2'] = ''
ROWS['time_sec'] = [parse_time_v1(t)[0] for t in ROWS['time_raw']]
ROWS['province_key'] = [province_key(p, ALIASES) for p in ROWS['province']]
v1r = ROWS['disposition'].eq('retained')
assert (ROWS.loc[v1r, 'event_date_v2'] == ROWS.loc[v1r, 'event_uid'].map(CLEAN_D)).all(), 'a v1 retained date changed'
assert (ROWS.loc[v1r, 'v2_disposition'] == 'retained').all(), 'a v1 retained row lost its disposition'
assert (ROWS.loc[ROWS['disposition'].eq('diagnostic'), 'v2_disposition'] == 'diagnostic').all()
_c = ROWS['reasons'].str.contains('id_conflict_other_fields') & ROWS['disposition'].eq('quarantined')
assert (ROWS.loc[_c, 'v2_disposition'] == 'quarantined').all() and (ROWS.loc[_c, 'v2_reasons'] == ROWS.loc[_c, 'reasons']).all(), 'strict v1 ID conflicts must stay'
assert not ROWS.loc[ROWS['v2_disposition'].isin(['retained', 'collapsed']), 'date_status'].ne('ok').where(~ROWS['date_decision'].eq(REPAIRED), False).any()
ret2 = ROWS[ROWS['v2_disposition'].eq('retained')]
assert ret2['time_sec'].notna().all() and ret2['event_date_v2'].ne('').all() and ret2['province_key'].ne('').all()

RECON = ROWS.groupby('source_year').agg(source_year_ce=('source_year_ce', 'first'), imported_rows=('source_row', 'size'),
                                        **{d: ('v2_disposition', lambda s, d=d: int((s == d).sum())) for d in DISPOSITIONS}).reset_index()
RECON['v1_retained'] = RECON['source_year'].map(ROWS[v1r].groupby('source_year').size()).fillna(0).astype(int)
RECON['retained_delta'] = RECON['retained'] - RECON['v1_retained']
RECON['manifest_rows'] = RECON['source_year'].map({str(f['be_year']): int(f['rows']) for f in V1RAW['files']})
RECON['balanced'] = (RECON[list(DISPOSITIONS)].sum(axis=1) == RECON['imported_rows']) & (RECON['imported_rows'] == RECON['manifest_rows'])
assert RECON['balanced'].all()
POLICY = ROWS.groupby(['source_year_ce', 'date_decision', 'v2_disposition']).size().rename('n_rows').reset_index()
TRANS = ROWS.groupby(['source_year_ce', 'disposition', 'v2_disposition']).size().rename('n_rows').reset_index()
CTXC = ROWS[ROWS['ctx10_status'].ne('not_evaluated')].groupby(['source_year_ce', 'ev09_context_status', 'ctx10_status']).size().rename('n_rows').reset_index()
LINEAGE_COLS = ['event_uid', 'source_file', 'source_year', 'source_year_ce', 'source_row', 'acc_code', 'disposition', 'reasons', 'date_raw', 'date_status',
                'date_candidates', 'ev09_status', 'ev09_context_status', 'ctx10_status', 'date_decision', 'repaired_date', 'date_confidence',
                'v2_disposition', 'v2_reasons', 'v2_kept_source_row', 'v2_redecided', 'event_date_v2', 'time_raw', 'time_sec', 'province', 'province_key',
                'coord_status']
write_csv_gz(DATA / 'row_lineage_v2.csv.gz', ROWS[LINEAGE_COLS])
write_csv(REP / 'row_reconciliation_v2.csv', RECON)
write_csv(REP / 'row_policy_counts.csv', POLICY)
write_csv(REP / 'v1_v2_disposition_transitions.csv', TRANS)
write_csv(REP / 'context_check_counts.csv', CTXC)
RUN['sections']['rows'] = {'imported': int(RECON['imported_rows'].sum()), 'v1_retained': int(v1r.sum()), 'v2_retained': int(len(ret2)),
                           'repaired_rows': int(ROWS['date_decision'].eq(REPAIRED).sum()), 'redecided_rows': int(ROWS['v2_redecided'].sum()),
                           'ctx10_counts': ROWS.loc[ROWS['ctx10_status'].ne('not_evaluated'), 'ctx10_status'].value_counts().to_dict(),
                           'xlsx_context_meta': CTX_META}
print(RECON.to_string(index=False))
print(RUN['sections']['rows'])


source_year  source_year_ce  imported_rows  retained  collapsed  quarantined  diagnostic  v1_retained  retained_delta  manifest_rows  balanced
       2564            2021          20062     19996          0           66           0        12599            7397          20062      True
       2565            2022          20566     20492          0           74           0        13249            7243          20566      True
       2566            2023          23841     20490          0         3351           0        13118            7372          23841      True
       2567            2024          23952     21788          0         2164           0        21788               0          23952      True
       2568            2025          23715     23542          0          173           0        23542               0          23715      True
       2569            2026          15239         0          0            0       15239            0               0          15239      True

## C. ธง unknown label

ทุกแถว quarantined/diagnostic ธงวันที่ผู้สมัครของตัวเอง (แถวที่ซ่อมแล้วแต่ยังถูกกักกันด้วยเหตุผลอื่น ธงเฉพาะวันที่ที่ซ่อม; แถวที่ซ่อมได้และ retained ไม่ธงอีก) ให้จังหวัดของตน หรือ **ทั้ง 77 จังหวัด** เมื่อจังหวัดว่าง/ไม่อยู่ใน panel. ID ที่ขัดแย้งยังธงทุกวันที่ผู้สมัคร. แถวที่ไม่มีวันที่ผู้สมัครเลยนับต่อจังหวัด-ปี (`v1_count_only` ตาม v1; ตัวเลือก `flag_province_year` มีใน config). ธงที่หายไปจาก v1 ต้องอธิบายได้ด้วยแถวที่ซ่อม (assert). ธง/mask ไม่ใช่ feature

In [6]:
# C. Unknown-label flags rebuilt from the REMAINING uncertainty (never realized-outcome features)
PSET = set(PROVINCES)
fx = ROWS[ROWS['v2_disposition'].isin(['quarantined', 'diagnostic'])].copy()
fx['flag_dates'] = np.where(fx['date_decision'].eq(REPAIRED), fx['repaired_date'], fx['date_candidates'])
fx['flag_kind'] = fx['v2_disposition']
fx['reasons'] = fx['v2_reasons']
# retained rows whose province is not one of the 77 keys cannot be placed: their date is flagged for ALL provinces (never silently dropped)
unres = ROWS[ROWS['v2_disposition'].eq('retained') & ~ROWS['province_key'].isin(PSET)].copy()
unres = unres.assign(flag_dates=unres['event_date_v2'], flag_kind='retained_province_unresolved', reasons='province_not_in_panel')
YEAR_DAYS = {y: [d for d in DATES if d.startswith(str(y))] for y in P['label_years_ce']}
FLAGS, UNPLACEABLE, FLAG_STATS = unknown_flags(pd.concat([fx, unres]), PROVINCES, DATES, YEAR_DAYS, R['no_candidate_date_policy'], P['all_provinces_token'])
FLAG_STATS.update(retained_province_unresolved=len(unres), policy=R['no_candidate_date_policy'],
                  rows_without_candidate_by_year=fx[fx['flag_dates'].eq('')].groupby('source_year_ce').size().to_dict())
# compare with v1 flags (expanded *ALL*), inside the panel dates
v1set = {k for k in expand_v1_flags(V1FLAGS, PROVINCES, P['all_provinces_token']) if k[1] in set(DATES)}
v2set = set(zip(FLAGS['province_key'], FLAGS['date']))
removed, added = v1set - v2set, v2set - v1set
rep_keys = set()
for pk, c in zip(ROWS.loc[ROWS['date_decision'].eq(REPAIRED), 'province_key'], ROWS.loc[ROWS['date_decision'].eq(REPAIRED), 'date_candidates']):
    rep_keys.update(((q, d) for q in (PROVINCES if pk not in PSET else [pk]) for d in c.split(';') if d))
assert removed <= rep_keys, 'a v1 flag disappeared without a repaired row explaining it'
if added:
    GAPS.append(f'{len(added)} province-days flagged in v2 but not in v1 (e.g. new conflicts after repair or unresolved provinces): {sorted(added)[:5]}')
_yr = lambda s: pd.Series([d[:4] for _, d in s], dtype=object).value_counts().sort_index()
FLAGCMP = pd.DataFrame({'v1_flagged_province_days': _yr(v1set), 'v2_flagged_province_days': _yr(v2set),
                        'removed_by_repair': _yr(removed), 'added_in_v2': _yr(added)}).fillna(0).astype(int).rename_axis('year').reset_index()
write_csv_gz(DATA / 'unknown_flags_v2.csv.gz', FLAGS)
write_csv(REP / 'flag_comparison_v1_v2.csv', FLAGCMP)
write_csv(REP / 'unplaceable_rows_province_year.csv', UNPLACEABLE)
RUN['sections']['flags'] = dict(FLAG_STATS, v1_flagged=len(v1set), v2_flagged=len(v2set), removed=len(removed), added=len(added))
print(FLAGCMP.to_string(index=False))
print(RUN['sections']['flags'])


year  v1_flagged_province_days  v2_flagged_province_days  removed_by_repair  added_in_v2
2021                      6406                       209               6197            0
2022                      6373                       219               6154            0
2023                      7779                      2381               5398            0
2024                     20297                     20297                  0            0
2025                       523                       523                  0            0
{'rows_in': 21067, 'rows_all_provinces': 1953, 'dates_outside_panel': 21273, 'rows_without_candidate_date': 0, 'flagged_province_days': 23629, 'retained_province_unresolved': 0, 'policy': 'v1_count_only', 'rows_without_candidate_by_year': {}, 'v1_flagged': 41378, 'v2_flagged': 23629, 'removed': 17749, 'added': 0}


## D. Panel รายวันและราย 6 ชั่วโมง

ตาราง 77 × 1826 วัน (2021-01-01..2025-12-31) และ × 4 บล็อก [00-06), [06-12), [12-18), [18-24) เวลาท้องถิ่น; เหตุการณ์ 06:00:00 อยู่ 06-12. issue = วันก่อนหน้า 18:00 (+07:00) ทุกบล็อก. ตรวจ: จำนวนไม่ติดลบ, รายวัน = ผลรวม 4 บล็อก, รวม = เหตุการณ์ที่วาง, ธงกระจายทุกบล็อก. เซลล์ว่างที่ทราบ = 0 บันทึกที่ถูกเลือก ไม่ใช่ศูนย์จริงที่พิสูจน์แล้ว

In [7]:
# D. Panels: 77 provinces x 1826 days, and the same x 4 local 6 h blocks (labels from each event's own time only)
EVENTS = ROWS[ROWS['v2_disposition'].eq('retained') & ROWS['province_key'].isin(PSET)].copy()
EVENTS = EVENTS[EVENTS['event_date_v2'].isin(set(DATES))]
assert len(EVENTS) == int(ROWS['v2_disposition'].eq('retained').sum()) - len(unres) - int(
    (ROWS['v2_disposition'].eq('retained') & ROWS['province_key'].isin(PSET) & ~ROWS['event_date_v2'].isin(set(DATES))).sum())
EVENTS['time_sec'] = EVENTS['time_sec'].astype(int)
EVENTS['time_block'] = [block_label(time_block_index(t, P['block_hours']), P['block_hours']) for t in EVENTS['time_sec']]
EV_OUT = EVENTS[['event_uid', 'source_year_ce', 'event_date_v2', 'time_sec', 'time_block', 'province_key', 'date_decision', 'date_confidence']].rename(
    columns={'event_date_v2': 'event_date'})
write_csv_gz(DATA / 'events_v2.csv.gz', EV_OUT)
DAILY, BLOCK = build_panels(EV_OUT, PROVINCES, DATES, FLAGS, UNPLACEABLE, P['splits'], P['issue_hour_local'], P['issue_days_before_target'],
                            P['utc_offset_seconds'], P['block_hours'])
PANEL_CHECKS = reconcile_panels(DAILY, BLOCK, len(EV_OUT))
assert len(DAILY) == P['expected_provinces'] * P['expected_days'] and len(BLOCK) == 4 * len(DAILY)
assert set(BLOCK['time_block']) == set(P['time_blocks'])
DAILY['label_status'] = np.where(DAILY['training_eligible'], 'selected_recorded_count_known_cell', 'unknown_possibly_undercounted')
BLOCK['label_status'] = np.where(BLOCK['training_eligible'], 'selected_recorded_count_known_cell', 'unknown_possibly_undercounted')
write_csv_gz(DATA / 'panel_daily_v2.csv.gz', DAILY)
write_csv_gz(DATA / 'panel_6h_v2.csv.gz', BLOCK)
PANEL_SUMMARY = pd.concat([dispersion(DAILY, ['split']).assign(unit='province_day'), dispersion(BLOCK, ['split', 'time_block']).assign(unit='province_6h')])
write_csv(REP / 'panel_target_summary.csv', PANEL_SUMMARY)
RUN['sections']['panels'] = {'daily_rows': len(DAILY), 'block_rows': len(BLOCK), 'events_placed': len(EV_OUT), 'checks': PANEL_CHECKS,
                             'eligible_daily': int(DAILY['training_eligible'].sum()), 'eligible_block': int(BLOCK['training_eligible'].sum())}
print(PANEL_SUMMARY.to_string(index=False))


           split  n_cells  n_eligible  events_eligible     mean  variance  zero_share  n_nonzero  max  variance_to_mean         unit time_block
development_2024    28182        7885             6168 0.782245  1.717544    0.565250       3428   20          2.195660 province_day        NaN
reused_seen_2025    28105       27582            21603 0.783228  1.874875    0.570372      11850   21          2.393778 province_day        NaN
           train    84315       81506            56770 0.696513  1.428832    0.597833      32779   20          2.051407 province_day        NaN
development_2024    28182        7885             1181 0.149778  0.178350    0.871655       1012    5          1.190762  province_6h      00-06
development_2024    28182        7885             1698 0.215346  0.310292    0.830818       1334    8          1.440903  province_6h      06-12
development_2024    28182        7885             1926 0.244261  0.371328    0.816107       1450    7          1.520210  province_6h    

## E. Feature ที่รู้ก่อนเวลา issue

กลุ่มตาม `workflow/v2-feature-policy.md`: core (จังหวัด, วันในสัปดาห์, เดือน, วันที่, สุดสัปดาห์, sin/cos วันของปี), timeblock (เฉพาะหน่วย 6 ชม.), เทศกาล (สงกรานต์ 13-15 เม.ย., ปีใหม่ 31 ธ.ค.-1 ม.ค. ข้ามปี; ระยะมีเครื่องหมาย clip ±14 และหน้าต่าง 1-3/4-7 วัน), ปฏิทินธนาคาร ธปท. as-of (replay add/remove ตาม known_from + 1 วัน; ปีที่ไม่รู้ coverage = NaN), ฤดูกาลแบบ proxy เดือน, อากาศ JMA GSM previous_day2 (ชั่วโมงครบเท่านั้น + guard bound + 6 ชม. ≤ issue ตรวจซ้ำใน 10), จราจร DOH AADT (ปีสำรวจก่อนปีเป้าหมาย และ proxy available_from ≤ issue). ขาด = NaN (imputer train-only ภายหลัง); ทุกโมเดลใช้ keys ชุดเดียวกัน. ประวัติอุบัติเหตุ = **pending**

In [8]:
# E. Features known before the issue time (core date, festival, bank calendar as-of, season, archived weather, as-of traffic)
I = dict(issue_hour=P['issue_hour_local'], days_before=P['issue_days_before_target'])
FEAT_DEFS = FCFG['features']
ALLOW = {u: [f['name'] for f in FEAT_DEFS if u in f['panels']] for u in ('daily', '6h')}
FN, FS = FCFG['forbidden_feature_names'], FCFG['forbidden_feature_substrings']
for u in ALLOW:  # the allowlist itself must pass the guard
    validate_feature_columns(ALLOW[u], ALLOW[u], FN, FS)
F = FCFG['festivals']
DF_DATE = core_date_features(DATES)
DF_DATE = DF_DATE.merge(festival_features(DATES, SCFG['calendar']['recurring_fixed_festivals'], F['clip_days'], F['windows']), on='target_date')
BANK = bank_calendar_features(DATES, CALREC, SCFG['calendar']['sources'], **I)
DF_DATE = DF_DATE.merge(BANK[['target_date', 'bank_holiday_asof', 'bank_calendar_known_asof']], on='target_date')
DF_DATE = DF_DATE.merge(season_features(DATES, FCFG['season']['months']), on='target_date')
WB = FCFG['weather']['buffer_hours']
assert WB == SCFG['weather']['assumed_publication_buffer_hours'], 'weather buffer must match the 09 guard'
WXD, WX_AUDIT_D = weather_features(WX_D, CODE_TO_KEY, WB, utc_offset=P['utc_offset_seconds'], prefix='wx_', **I)
WXB, WX_AUDIT_B = weather_features(WX_B, CODE_TO_KEY, WB, utc_offset=P['utc_offset_seconds'], prefix='wx_block_', **I)
TV = traffic_vintage_by_date(TMETA, DATES, utc_offset=P['utc_offset_seconds'], **I)
TRF = traffic_features(TSUM, TV, PROVINCES)

keys_d = DAILY[['province_key', 'target_date']].copy()
FD = keys_d.merge(DF_DATE, on='target_date', how='left').merge(
    WXD.drop(columns=['time_block']) if len(WXD) else WXD.drop(columns=['time_block']), on=['province_key', 'target_date'], how='left')
FD = FD.merge(TRF.drop(columns=['trf_vintage_survey_year_ce', 'trf_vintage_available_from_utc']), on=['province_key', 'target_date'], how='left')
FD.insert(2, 'province', FD['province_key'])
keys_b = BLOCK[['province_key', 'target_date', 'time_block']].copy()
FB = keys_b.merge(FD.drop(columns=['province']), on=['province_key', 'target_date'], how='left')
FB = FB.merge(WXB, on=['province_key', 'target_date', 'time_block'], how='left')
FB.insert(3, 'province', FB['province_key'])
FEAT_D, FEAT_B = ALLOW['daily'], ALLOW['6h']
for frame, cols, keys in ((FD, FEAT_D, ['province_key', 'target_date']), (FB, FEAT_B, ['province_key', 'target_date', 'time_block'])):
    for c in cols:  # an unavailable source still yields the column (all NaN), so every model sees the same keys and columns
        if c not in frame:
            frame[c] = np.nan
    validate_feature_columns(cols, cols, FN, FS)
    assert_finite_features(frame, cols)
    assert not frame.duplicated(keys).any()
FD = FD[['province_key', 'target_date'] + FEAT_D]
FB = FB[['province_key', 'target_date', 'time_block'] + [c for c in FEAT_B if c != 'time_block']]
assert len(FD) == len(DAILY) and len(FB) == len(BLOCK)
assert (FD[['province_key', 'target_date']].to_numpy() == DAILY[['province_key', 'target_date']].to_numpy()).all()
assert (FB[['province_key', 'target_date', 'time_block']].to_numpy() == BLOCK[['province_key', 'target_date', 'time_block']].to_numpy()).all()
write_csv_gz(DATA / 'features_daily_v2.csv.gz', FD)
write_csv_gz(DATA / 'features_6h_v2.csv.gz', FB)
# per-date source audit columns (NOT features): which traffic vintage / calendar coverage applied
SRC_AUDIT = TV.merge(BANK[['target_date', 'bank_calendar_coverage']], on='target_date')
write_csv_gz(DATA / 'feature_source_audit_by_date.csv.gz', SRC_AUDIT)
RUN['sections']['features'] = {'daily_features': FEAT_D, 'block_features': FEAT_B, 'weather_daily': WX_AUDIT_D, 'weather_6h': WX_AUDIT_B,
                               'traffic_vintages_used': TV['trf_vintage_survey_year_ce'].value_counts(dropna=False).to_dict(),
                               'history': FCFG['pending_groups']['history']}
print(RUN['sections']['features'])


{'daily_features': ['province', 'day_of_week', 'month', 'day_of_month', 'is_weekend', 'doy_sin', 'doy_cos', 'fest_songkran_dist', 'fest_songkran_in', 'fest_songkran_pre_1_3', 'fest_songkran_pre_4_7', 'fest_songkran_post_1_3', 'fest_songkran_post_4_7', 'fest_new_year_dist', 'fest_new_year_in', 'fest_new_year_pre_1_3', 'fest_new_year_pre_4_7', 'fest_new_year_post_1_3', 'fest_new_year_post_4_7', 'bank_holiday_asof', 'bank_calendar_known_asof', 'season_hot', 'season_wet', 'season_cool', 'wx_rain_sum_mm_d2', 'wx_temp_mean_c_d2', 'trf_mean_aadt', 'trf_median_aadt', 'trf_mean_heavy_share_pct', 'trf_n_station_rows'], 'block_features': ['province', 'day_of_week', 'month', 'day_of_month', 'is_weekend', 'doy_sin', 'doy_cos', 'time_block', 'fest_songkran_dist', 'fest_songkran_in', 'fest_songkran_pre_1_3', 'fest_songkran_pre_4_7', 'fest_songkran_post_1_3', 'fest_songkran_post_4_7', 'fest_new_year_dist', 'fest_new_year_in', 'fest_new_year_pre_1_3', 'fest_new_year_pre_4_7', 'fest_new_year_post_1_3', 

## F. Matrix ความพร้อมของ feature และ leakage audit

`feature_availability_matrix.csv` (group/name/unit/source/available_at/known_before_issue/test_status/assumptions), allowlist รายวัน/6 ชม. และรายชื่อต้องห้าม. Negative controls ที่ตั้งใจให้รั่ว (จราจรปีอนาคต, วันหยุดพิเศษประกาศช้า + การลบ, ปฏิทินไม่ทราบ ≠ วันปกติ, ชั่วโมงอากาศหลัง issue, guard 09 ที่ถูกแก้, คอลัมน์ผลลัพธ์/ป้าย/mask/ID) ต้องถูกปฏิเสธทั้งหมด

In [9]:
# F. Feature availability matrix, as-of assertions on the real matrices and deliberately leaky negative controls
CONTROLS = leakage_negative_controls(SCFG['calendar']['sources'], ALLOW['6h'], FN, FS, WB, utc_offset=P['utc_offset_seconds'], **I)
assert all(c['rejected'] for c in CONTROLS), [c for c in CONTROLS if not c['rejected']]
# as-of assertions on the real outputs
ASOF = {}
_tv = TV.dropna(subset=['trf_vintage_survey_year_ce'])
_iss = (pd.to_datetime(_tv['target_date']) - pd.Timedelta(days=I['days_before']) + pd.Timedelta(hours=I['issue_hour'])
        - pd.Timedelta(seconds=P['utc_offset_seconds'])).dt.tz_localize('UTC')
ASOF['traffic_prior_year_only'] = bool((_tv['trf_vintage_survey_year_ce'].astype(int) < pd.to_datetime(_tv['target_date']).dt.year).all())
ASOF['traffic_available_before_issue'] = bool((pd.to_datetime(_tv['trf_vintage_available_from_utc'], utc=True) <= _iss).all())
_cal = CALREC[CALREC['applied']]
_late = []
for d, hol in zip(BANK['target_date'], BANK['bank_holiday_asof']):
    if hol == 1.0:
        issue = dt.datetime.combine(dt.date.fromisoformat(d) - dt.timedelta(days=I['days_before']), dt.time(I['issue_hour']))
        adds = _cal[(_cal['date'] == d) & _cal['action'].eq('add')]
        if not (adds['known_from'].map(available_from_local) <= issue).any():
            _late.append(d)
ASOF['bank_holiday_known_before_issue'] = not _late
ASOF['bank_unknown_coverage_is_nan'] = bool(BANK.loc[BANK['bank_calendar_known_asof'] == 0, 'bank_holiday_asof'].isna().all())
ASOF['weather_guard_rechecked_in_10'] = True  # weather_features raises on any 09/10 guard disagreement
ASOF['features_have_no_infinity'] = bool(assert_finite_features(FD, FEAT_D) and assert_finite_features(FB, FEAT_B))
ASOF['same_keys_all_models'] = True  # asserted in E: feature rows == panel rows in identical order
assert all(ASOF.values()), ASOF
cov_d = FD.assign(year=FD['target_date'].str[:4])
MATRIX = []
for f in FEAT_DEFS:
    nn = float(cov_d[f['name']].notna().mean()) if f['name'] in cov_d else (float(FB[f['name']].notna().mean()) if f['name'] in FB else 0.0)
    status = ('deterministic_rule_tested' if f['available_at'] in ('deterministic', 'static') else
              ('asof_assertions_and_negative_controls_passed' if nn > 0 else 'source_unavailable_all_nan'))
    MATRIX.append({'group': f['group'], 'name': f['name'], 'unit': f['unit'], 'source': f['source'], 'available_at': f['available_at'],
                   'known_before_issue': True, 'panels': ';'.join(f['panels']), 'non_null_share': nn, 'test_status': status,
                   'assumptions': f['assumptions']})
MATRIX.append({'group': 'history', 'name': '(none)', 'unit': '', 'source': 'v1/v2 labels', 'available_at': 'unverified', 'known_before_issue': False,
               'panels': '', 'non_null_share': 0.0, 'test_status': 'pending_not_implemented', 'assumptions': FCFG['pending_groups']['history']})
MATRIX = pd.DataFrame(MATRIX)
write_csv(REP / 'feature_availability_matrix.csv', MATRIX)
LEAK = {'production_ready': FCFG['readiness']['production_ready'], 'operational_status': FCFG['readiness']['operational_status'],
        'caveats': FCFG['readiness']['caveats'], 'allowlist_daily': FEAT_D, 'allowlist_6h': FEAT_B, 'forbidden_feature_names': FN,
        'forbidden_feature_substrings': FS, 'never_model_inputs': ['target_date', 'split', 'event_uid', 'acc_code', 'y_recorded_count',
        'training_eligible', 'possibly_undercounted', 'flag_reasons', 'n_flagged_rows', 'unplaceable_rows_province_year', 'label_status'],
        'asof_assertions': ASOF, 'negative_controls': CONTROLS, 'weather_lead_audit': {'daily': WX_AUDIT_D, 'six_hour': WX_AUDIT_B},
        'history': FCFG['pending_groups']['history']}
write_json(REP / 'leakage_audit.json', LEAK)
print(MATRIX[['group', 'name', 'non_null_share', 'test_status']].to_string(index=False))
print('negative controls rejected:', sum(c['rejected'] for c in CONTROLS), '/', len(CONTROLS))


        group                     name  non_null_share                                  test_status
         core                 province        1.000000                    deterministic_rule_tested
         core              day_of_week        1.000000                    deterministic_rule_tested
         core                    month        1.000000                    deterministic_rule_tested
         core             day_of_month        1.000000                    deterministic_rule_tested
         core               is_weekend        1.000000                    deterministic_rule_tested
         core                  doy_sin        1.000000                    deterministic_rule_tested
         core                  doy_cos        1.000000                    deterministic_rule_tested
    timeblock               time_block        1.000000                    deterministic_rule_tested
     festival       fest_songkran_dist        1.000000                    deterministic_rule_tested


## G. สถิติเชิงพรรณนา

จำนวน **บันทึกที่ถูกเลือก** ก่อน/หลังซ่อมตามปีและวันที่ 1-12 vs 13+, เดือน, วันในสัปดาห์, บล็อกเวลา, จังหวัด; การกระจายของเป้าหมาย (ศูนย์/ไม่ศูนย์/variance-to-mean) ใน panel_target_summary; ความครอบคลุมของแหล่ง feature รายปี, lead ของอากาศ, vintage จราจร, coverage ปฏิทินธนาคาร. กราฟไม่ใช่ความเสี่ยงหรือผลเชิงสาเหตุ; ประเภทอุบัติเหตุเป็น descriptive เท่านั้น

In [10]:
# G. Descriptive analytics of RECORDED SELECTED counts (not risk, not causal feature effects) + modest PNGs
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

v1ev = ROWS[ROWS['disposition'].eq('retained')].assign(event_date=lambda d: d['event_date_v2'])
v2ev = EV_OUT
def _dom(df, label):
    d = pd.to_datetime(df['event_date'])
    return pd.DataFrame({'year': d.dt.year, 'dom_bucket': np.where(d.dt.day <= 12, 'day_1_12', 'day_13_plus')}).value_counts().rename(label)
DOM = pd.concat([_dom(v1ev, 'v1_retained'), _dom(v2ev, 'v2_retained')], axis=1).fillna(0).astype(int).reset_index().sort_values(['year', 'dom_bucket'])
DOM['delta'] = DOM['v2_retained'] - DOM['v1_retained']
write_csv(REP / 'descriptive_day_of_month_v1_v2.csv', DOM)
_d = pd.to_datetime(v2ev['event_date'])
DESC = {'month': _d.dt.month, 'weekday': _d.dt.dayofweek, 'time_block': v2ev['time_block']}
for k, s in DESC.items():
    t = pd.crosstab(_d.dt.year, s).rename_axis(index='year').reset_index()
    t.columns = [str(c) for c in t.columns]
    write_csv(REP / f'descriptive_events_by_year_{k}.csv', t)
write_csv(REP / 'descriptive_events_by_province_year.csv',
          pd.crosstab(v2ev['province_key'], _d.dt.year).rename_axis(index='province_key').reset_index().rename(columns=str))
# coverage of feature groups by year (non-null share) and eligibility
_cv = FD.assign(year=FD['target_date'].str[:4], eligible=DAILY['training_eligible'].to_numpy())
COV = _cv.groupby('year').agg(cells=('province_key', 'size'), eligible_share=('eligible', 'mean'),
                              weather_rain_nonnull=('wx_rain_sum_mm_d2', lambda s: s.notna().mean()),
                              weather_temp_nonnull=('wx_temp_mean_c_d2', lambda s: s.notna().mean()),
                              traffic_nonnull=('trf_mean_aadt', lambda s: s.notna().mean()),
                              bank_calendar_known=('bank_calendar_known_asof', 'mean')).reset_index()
_cb = FB.assign(year=FB['target_date'].str[:4])
COV['weather_block_rain_nonnull'] = COV['year'].map(_cb.groupby('year')['wx_block_rain_sum_mm_d2'].apply(lambda s: s.notna().mean()))
write_csv(REP / 'feature_coverage_by_year.csv', COV)
TVC = TV.assign(year=TV['target_date'].str[:4]).groupby(['year', TV['trf_vintage_survey_year_ce'].fillna(-1).astype(int)]).size().rename('n_dates').reset_index()
TVC = TVC.rename(columns={'trf_vintage_survey_year_ce': 'survey_year_ce_used_or_minus1_none'})
write_csv(REP / 'traffic_vintage_by_target_year.csv', TVC)
BANKC = BANK.assign(year=BANK['target_date'].str[:4]).groupby(['year', 'bank_calendar_coverage']).size().rename('n_dates').reset_index()
write_csv(REP / 'bank_calendar_coverage_by_year.csv', BANKC)
write_csv(REP / 'weather_lead_bounds.csv', pd.DataFrame([dict(unit='daily', **WX_AUDIT_D), dict(unit='6h', **WX_AUDIT_B)]))
# descriptive-only outcome illustration (accident type) for v1-retained rows: NEVER a predictor
OUTC = read_str_csv(V1 / 'data' / 'interim' / 'events_descriptive_outcomes.csv')
_at = OUTC.merge(ROWS[['event_uid', 'source_year_ce']].drop_duplicates('event_uid'), on='event_uid', how='left')
_at['accident_type'] = _at['accident_type'].map(norm_text).replace('', '(blank)')
AT = _at.groupby(['source_year_ce', 'accident_type']).size().rename('n_events').reset_index().sort_values(['source_year_ce', 'n_events'], ascending=[True, False])
write_csv(REP / 'descriptive_only_accident_type_by_year.csv', AT)
# PNGs (English labels: the headless kernel may lack Thai fonts)
fig, ax = plt.subplots(figsize=(8, 3.5))
piv = DOM.pivot(index='year', columns='dom_bucket', values='delta').reindex(columns=['day_1_12', 'day_13_plus']).fillna(0)
piv.plot.bar(ax=ax)
ax.set_title('Retained recorded events: v2 minus v1 by day-of-month bucket (date repair effect)')
ax.set_ylabel('selected recorded events')
save_png(fig, REP / 'date_repair_by_day_of_month.png')
plt.close(fig)
fig, ax = plt.subplots(figsize=(8, 3.5))
pd.read_csv(REP / 'descriptive_events_by_year_time_block.csv').set_index('year').plot.bar(ax=ax)
ax.set_title('Selected recorded events by local 6 h block and year (labels, not risk)')
save_png(fig, REP / 'events_by_time_block.png')
plt.close(fig)
fig, ax = plt.subplots(figsize=(8, 3.5))
COV.set_index('year')[['eligible_share', 'weather_rain_nonnull', 'traffic_nonnull', 'bank_calendar_known']].plot.bar(ax=ax, ylim=(0, 1.05))
ax.set_title('Share of province-days: eligible label / feature source available at issue')
save_png(fig, REP / 'source_coverage_by_year.png')
plt.close(fig)
print(DOM.to_string(index=False))
print(COV.to_string(index=False))


 year  dom_bucket  v1_retained  v2_retained  delta
 2021 day_13_plus        11703        11703      0
 2021    day_1_12          896         8293   7397
 2022 day_13_plus        12389        12389      0
 2022    day_1_12          860         8103   7243
 2023 day_13_plus        12303        12303      0
 2023    day_1_12          815         8187   7372
 2024 day_13_plus        12903        12903      0
 2024    day_1_12         8885         8885      0
 2025 day_13_plus        14168        14168      0
 2025    day_1_12         9374         9374      0
year  cells  eligible_share  weather_rain_nonnull  weather_temp_nonnull  traffic_nonnull  bank_calendar_known  weather_block_rain_nonnull
2021  28105        0.992564              0.999680              1.000000         0.000000             0.980822                    0.999875
2022  28105        0.992208              0.998612              1.000000         0.769863             1.000000                    0.999600
2023  28105        0.9152

## H. รายงานไทยและ manifest

`reports/v2/preparation/data_preparation_v2_th.md` และ `preparation_manifest.json` (hash ของทุก input, FUNCTIONS/code cells, output, และ `next_stage_interface` สำหรับ notebook 11). ไฟล์ขนาดใหญ่อยู่ใน `data/v2` (gitignored)

In [11]:
# H. Thai report (deterministic: no timestamps) and preparation manifest with the interface for notebook 11
rs, fl, pn = RUN['sections']['rows'], RUN['sections']['flags'], RUN['sections']['panels']
AL = V1CFG['schema_aliases']
ROLE = {'date_raw': 'label (วันที่เหตุการณ์) + date repair', 'time_raw': 'label (บล็อกเวลา 6 ชม.) + context check', 'province': 'label key + context check',
        'acc_code': 'lineage/ID group only', 'latitude': 'v1 coordinate exclusion only', 'longitude': 'v1 coordinate exclusion only',
        'year_reported': 'v1 consistency check only'}
md = ['# รายงานเตรียมข้อมูล v2 (Notebook 10): ซ่อมวันที่, panel เป้าหมาย, feature และตรวจ leakage', '',
      'สถานะ: งานวิจัยเท่านั้น | production_ready = false | operational = BLOCKED | **ยังไม่มีการ fit หรือเทียบโมเดลใด ๆ** (ขั้นถัดไปคือ notebook 11)', '',
      '## 1. สิ่งที่เปลี่ยนจาก v1 และเหตุผล', '',
      f"- แถวนำเข้า {rs['imported']} แถว (CSV ดิบเดิมทุกปี รวม 2026 ที่เป็น diagnostic เท่านั้น); v1 retained {rs['v1_retained']} -> v2 retained {rs['v2_retained']}",
      f"- ซ่อมวันที่กำกวม (D/M vs M/D) ได้ {rs['repaired_rows']} แถว โดยใช้เฉพาะคู่ข้อมูล XLSX ของผู้เผยแพร่ที่ ACC_CODE ตรงตัว, วันที่ typed เดียว, "
      'อยู่ในชุดวันที่ผู้สมัครของแถวนั้น และ notebook 10 ตรวจบริบทจังหวัด + เวลา (HH:MM) ซ้ำเองอย่างอิสระแล้วตรงกันทุกแถวของ ID',
      f"- ผลตรวจบริบทอิสระ: {rs['ctx10_counts']} (unavailable = ข้อมูลบริบทไม่ครบ จึงไม่ถือว่ายืนยัน และไม่ซ่อม)",
      '- วันที่ที่ v1 อ่านได้ไม่กำกวมไม่ถูกเปลี่ยนเลย (ตรวจ assert แล้ว); กลุ่ม id_conflict_* เดิมยังถูกกักกัน; ไม่มีแถวใหม่จาก XLSX; '
      'ปี 2022 เป็นรูปแบบผสม จึงไม่มี parser D/M หรือ M/D แบบรวมทั้งปี', '',
      '| ปี พ.ศ. | ปี ค.ศ. | นำเข้า | retained | collapsed | quarantined | diagnostic | v1 retained | ส่วนต่าง | สมดุล |', '|---|---|---|---|---|---|---|---|---|---|']
md += [f'| {r.source_year} | {r.source_year_ce} | {r.imported_rows} | {r.retained} | {r.collapsed} | {r.quarantined} | {r.diagnostic} | {r.v1_retained} | '
       f'{r.retained_delta} | {r.balanced} |' for r in RECON.itertuples()]
md += ['', '## 2. ธงป้ายกำกับที่ไม่ทราบ (unknown labels)', '',
       f"ธงสร้างใหม่จากความไม่แน่นอนที่เหลือ: v1 {fl['v1_flagged']} -> v2 {fl['v2_flagged']} province-days (ลบเพราะซ่อมได้ {fl['removed']}, เพิ่ม {fl['added']}). "
       f"แถวจังหวัดว่าง/ไม่อยู่ใน panel ธงทั้ง 77 จังหวัด ({fl['rows_all_provinces']} แถว); แถวไม่มีวันที่ผู้สมัคร {fl['rows_without_candidate_date']} แถว "
       f"(นโยบาย {fl['policy']}: นับใน unplaceable_rows_province_year เป็น caveat ไม่ใช่ศูนย์ที่ทราบ). เซลล์ที่ถูกธง training_eligible = false และธงรายวันกระจายไปทั้ง 4 บล็อก. "
       'ธง/mask เป็นตัวกรองวิจัยแบบ offline ที่ได้จากผลลัพธ์ ห้ามใช้เป็น feature', '',
       '## 3. Panel เป้าหมาย', '',
       f"- province-day: {pn['daily_rows']} แถว (77 x 1826), eligible {pn['eligible_daily']}; province x 6 ชม.: {pn['block_rows']} แถว, eligible {pn['eligible_block']}",
       f"- เหตุการณ์ที่วางลง panel {pn['events_placed']}; ตรวจ: {pn['checks']}",
       '- issue = วันก่อนเป้าหมาย 18:00 Asia/Bangkok ทั้งสองหน่วย; target_start/target_end ระบุ offset +07:00 ชัดเจน; 00-06/06-12/12-18/18-24 (06:00:00 อยู่บล็อก 06-12; 24:00 ไม่ valid)',
       '- y_recorded_count = จำนวนบันทึก MOT ที่ถูกเลือก ไม่ใช่อุบัติเหตุจริงทั้งหมด; ศูนย์ในเซลล์ที่ทราบ = ไม่มีบันทึกที่ถูกเลือก ไม่ใช่ศูนย์จริงที่พิสูจน์แล้ว',
       '- split เดิม: train 2021-2023, development_2024, reused_seen_2025 (เคยเห็นแล้ว ไม่ใช่ test อิสระ); 2026 ไม่เป็น label', '',
       '## 4. คอลัมน์ canonical ที่ใช้/ไม่ใช้', '', '| canonical | header ต้นฉบับ (alias) | บทบาทใน v2 |', '|---|---|---|']
md += [f"| {k} | {' / '.join(v)} | {ROLE.get(k, 'ไม่ใช้เป็น feature (descriptive/v1 ID-conflict signature เท่านั้น)')} |" for k, v in AL.items()]
md += ['', 'การ normalize ข้ามปี: ข้อความถูกยุบช่องว่างเท่านั้น (ไม่ recode ประเภทอุบัติเหตุ); ชื่อจังหวัดใช้ alias ที่ตรวจแล้วแบบตรงตัว '
       f"({'; '.join(f'{k} -> {v}' for k, v in ALIASES.items())}); ประเภทอุบัติเหตุแสดงเฉพาะใน descriptive_only_accident_type_by_year.csv (แถว v1 retained) และไม่ใช่ตัวทำนาย", '',
       '## 5. Feature (รู้ก่อนเวลา issue) และข้อจำกัด', '', '| กลุ่ม | ชื่อ | หน่วย | available_at | non-null | สถานะทดสอบ |', '|---|---|---|---|---|---|']
md += [f'| {r.group} | {r.name} | {r.unit} | {r.available_at} | {r.non_null_share:.3f} | {r.test_status} |' for r in MATRIX.itertuples()]
md += ['', f"- เทศกาล: {F['sign_convention']}", f"- ฤดูกาล: {FCFG['season']['note']}", f"- อากาศ: {FCFG['weather']['note']}", f"- จราจร: {FCFG['traffic']['note']}",
       '- ปฏิทินธนาคาร (ธปท.) ≠ วันหยุดราชการ; ปีที่ไม่รู้ coverage ณ เวลา issue เป็น NaN ไม่ใช่วันปกติ',
       f"- ประวัติอุบัติเหตุ: {FCFG['pending_groups']['history']}",
       f"- negative controls ที่ถูกปฏิเสธ: {sum(c['rejected'] for c in CONTROLS)}/{len(CONTROLS)}; as-of assertions: {ASOF}", '',
       '## 6. ความครอบคลุมของแหล่งข้อมูลรายปี', '', COV.round(4).to_markdown(index=False) if hasattr(COV, 'to_markdown') and False else COV.round(4).to_csv(index=False), '',
       '## 7. ข้อจำกัด / ช่องว่าง (จากการรันจริง)', '']
md += [f'- {c}' for c in FCFG['readiness']['caveats']] + [f'- {g}' for g in GAPS]
md += ['', '## 8. ส่งต่อให้ notebook 11', '',
       '- ใช้ panel_daily_v2 / panel_6h_v2 + features_daily_v2 / features_6h_v2 (keys เรียงตรงกันทุกแถว) จาก data/v2 ที่ hash ตรงกับ preparation_manifest.json',
       '- train/evaluate เฉพาะ training_eligible = true; ทุกโมเดลและ baseline ใช้ keys ชุดเดียวกัน; feature ที่หายเป็น NaN ให้ imputer ที่ fit จาก train เท่านั้น',
       '- ห้ามนำ target_date/split/ID/target/mask/ธง เข้าโมเดล; ต้องล็อก protocol v2 ก่อน fit; ยังไม่มีข้ออ้างว่าโมเดลใดดีกว่า baseline']
(REP / 'data_preparation_v2_th.md').write_text('\n'.join(md) + '\n', 'utf-8')

DATA_FILES = ['row_lineage_v2.csv.gz', 'unknown_flags_v2.csv.gz', 'events_v2.csv.gz', 'panel_daily_v2.csv.gz', 'panel_6h_v2.csv.gz',
              'features_daily_v2.csv.gz', 'features_6h_v2.csv.gz', 'feature_source_audit_by_date.csv.gz']
fn10 = functions_cells_source(NB10)
MANIFEST = {
    'producer': FCFG['producer'], 'schema': 'roadaware-v2-preparation-manifest/1', 'status': 'completed' if not GAPS else 'completed_with_gaps',
    'status_note': 'preparation/audit only; no model fitted; passing checks is not acceptance', 'started_utc': RUN['started_utc'], 'finished_utc': utc_now(),
    'production_ready': False, 'operational_status': 'BLOCKED', 'functions_cells_sha256': sha256_bytes(fn10.encode('utf-8')),
    'code_cells_sha256': code_cells_sha256(NB10), 'config_sha256': sha256_file(FCFG_PATH), 'inputs': INPUTS,
    'roots': {'v1': str(V1), 'external': str(EXT), 'sources_09': str(SRC), 'data': str(DATA), 'output': str(OUT)},
    'versions': {'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__},
    'sections': RUN['sections'], 'gaps': GAPS,
    'data_outputs': [{'path': n, 'sha256': sha256_file(DATA / n), 'bytes': (DATA / n).stat().st_size} for n in DATA_FILES],
    'report_outputs': [{'path': p.name, 'sha256': sha256_file(p)} for p in sorted(REP.iterdir()) if p.is_file() and p.name != 'preparation_manifest.json'],
    'deterministic_outputs_note': 'all data_outputs and report_outputs except PNG bytes (matplotlib-version dependent) are expected byte-identical on a rerun with identical inputs',
    'next_stage_interface': {
        'daily': {'panel': 'panel_daily_v2.csv.gz', 'features': 'features_daily_v2.csv.gz', 'keys': ['province_key', 'target_date'],
                  'target': 'y_recorded_count', 'mask': 'training_eligible', 'split': 'split', 'feature_columns': FEAT_D},
        '6h': {'panel': 'panel_6h_v2.csv.gz', 'features': 'features_6h_v2.csv.gz', 'keys': ['province_key', 'target_date', 'time_block'],
               'target': 'y_recorded_count', 'mask': 'training_eligible', 'split': 'split', 'feature_columns': FEAT_B},
        'feature_groups': {g: [f['name'] for f in FEAT_DEFS if f['group'] == g] for g in dict.fromkeys(f['group'] for f in FEAT_DEFS)},
        'categorical': ['province', 'time_block'], 'never_model_inputs': LEAK['never_model_inputs'], 'history': 'pending',
        'splits': P['splits']},
}
write_json(REP / 'preparation_manifest.json', MANIFEST)
print('reports ->', REP, '| data ->', DATA, '| gaps:', len(GAPS))


reports -> /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/reports/v2/preparation | data -> /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/data/v2 | gaps: 0


In [12]:
# Final checks: v1 untouched, read-only inputs unchanged, strict JSON, manifest hashes match the files on disk
V1_TREE_AFTER = protected_tree_hashes(V1_PROTECTED, V1)
changed = sorted(k for k in set(V1_TREE_BEFORE) | set(V1_TREE_AFTER) if V1_TREE_BEFORE.get(k) != V1_TREE_AFTER.get(k))
assert not changed, f'protected v1 files changed: {changed[:10]}'
for i in INPUTS:
    if Path(i['path']).is_file() and i['role'] not in ('config', '09_functions_cells'):
        assert sha256_file(i['path']) == i['sha256'], f"input changed during the run: {i['path']}"
for p in sorted(REP.glob('*.json')):
    json.loads(p.read_text('utf-8'), parse_constant=lambda c: (_ for _ in ()).throw(ValueError(f'{p.name}: non-strict JSON {c}')))
M = json.loads((REP / 'preparation_manifest.json').read_text('utf-8'))
assert all(sha256_file(DATA / o['path']) == o['sha256'] for o in M['data_outputs'])
assert all(sha256_file(REP / o['path']) == o['sha256'] for o in M['report_outputs'])
assert M['production_ready'] is False and M['operational_status'] == 'BLOCKED'
print('final checks passed: v1 unchanged, inputs unchanged, strict JSON, manifest hashes verified;',
      f"{len(M['data_outputs'])} data files, {len(M['report_outputs'])} report files; status={M['status']}")


final checks passed: v1 unchanged, inputs unchanged, strict JSON, manifest hashes verified; 8 data files, 24 report files; status=completed


## ข้อจำกัดและการส่งต่อ

- ยังไม่มีการ fit/เทียบโมเดล; ต้องล็อก protocol v2 ก่อน (Poisson/RandomForest/HGB เทียบ baseline ที่แข็งแรงบน keys เดียวกัน, ablation ตามกลุ่ม feature)
- 2024/2025 เคยเห็นแล้ว จึงเป็น diagnostic สำหรับพัฒนา ไม่ใช่ test อิสระ; cohort v2 ต่างจาก v1 จึงห้ามเทียบตัวเลขข้ามเวอร์ชันโดยไม่มี shared keys
- training_eligible ได้จากผลลัพธ์ (offline research filter) ใช้กับการพยากรณ์ล่วงหน้าจริงไม่ได้
- ประวัติอุบัติเหตุ (conditional history) ยัง pending จนกว่า Codex ยืนยัน scenario แยก